# Функции и ООП в Python: от `def` до маленькой программы

Продолжение ноутбука про контейнеры. Здесь две большие темы: **функции** (глава 4) и
**объектно-ориентированное программирование** (глава 5). Цель — не «познакомиться с синтаксисом»,
а научиться самостоятельно писать функции, классы и небольшие программы, в которых несколько
объектов работают вместе.

## Как устроен ноутбук

Каждая тема идёт по одной схеме: короткая теория → пример кода → пояснение, как Python его
выполняет → типичные ошибки → вопрос на понимание (ответ спрятан под спойлером) → мини-задача
с автопроверкой. После теории по функциям — 10 заданий по возрастанию сложности; после теории
по ООП — три уровня практики вокруг двух классов и итоговый проект.

## Как устроена проверка

Проверка та же, что в ноутбуке про контейнеры: файл `tests_functions_oop.py` должен лежать рядом
с ноутбуком, в ячейке задания уже есть шаблон, последняя строка ячейки — `test_<имя>(<имя>)`.
Отличие одно: раньше проверялся вывод программы, а теперь — **функции и классы**. Тест вызывает
вашу функцию с разными аргументами (или прогоняет сценарий с вашим классом) и сравнивает
результат с ожидаемым:

```
Задача 1. clamp — тестов: 9
  ✅ тест 1: clamp(5, 0, 10) — OK
  ❌ тест 2: clamp(-3, 0, 10) — неверный ответ
     ожидалось: 0
     получено: -3
Итог: пройдено 8 из 9 ❌
```

Три вещи, о которых стоит помнить:

- функции должны **возвращать** результат через `return`, а не печатать его — тест смотрит на
  возвращаемое значение;
- если в условии есть ограничение («используйте рекурсию», «нельзя `sorted()`»), проверка
  заглядывает в код решения и сообщает о нарушении значком ⛔;
- ошибка внутри вашего кода не роняет ноутбук: тест покажет тип ошибки и строку, где она случилась.

Примеры в условиях записаны как в интерактивной сессии Python: строка с `>>>` — вызов,
строка под ней — результат.

## Оглавление

**Глава 4. Функции**
- 4.1. Функции. Области видимости. Передача параметров
- 4.2. Позиционные и именованные аргументы. Функции высших порядков. Lambda
- 4.3. Рекурсия. Декораторы. Генераторы
- 4.4. Десять заданий на функции

**Глава 5. ООП и другие возможности Python**
- 5.1. Объектная модель Python. Классы, поля и методы
- 5.2. Магические методы, переопределение методов. Наследование
- 5.3. Исключения и модули
- 5.4. Уровень 1 — собираем класс `BankAccount`
- 5.5. Уровень 2 — класс `Bank`
- 5.6. Уровень 3 — наследование, отчёты и маленькая программа
- 5.7. Итоговый проект — библиотека

**Разбор решений** · **Что я теперь умею**


# Глава 4. Функции

## 4.1. Функции. Области видимости. Передача параметров

### Что такое функция и зачем она нужна

Функция — это именованный кусок кода, который можно вызывать много раз с разными данными.
Без функций программа быстро превращается в повторяющиеся блоки: посчитать площадь одного
прямоугольника, потом второго, потом третьего — и каждый раз копировать формулу. С функцией
формула записывается один раз, а дальше используется по имени.

Функции нужны для трёх вещей:

- **не повторяться** — одна формула в одном месте, где её можно исправить;
- **делить задачу на части** — большая программа становится набором маленьких понятных функций;
- **давать коду имена** — `is_leap_year(2024)` читается лучше, чем
  `2024 % 4 == 0 and (2024 % 100 != 0 or 2024 % 400 == 0)`.

In [1]:
# Без функции: формула повторяется, и в каждой копии можно ошибиться
print(3 * 4)
print(5 * 2)
print(7 * 7)

# С функцией: формула записана один раз
def rectangle_area(width, height):
    return width * height

print(rectangle_area(3, 4), rectangle_area(5, 2), rectangle_area(7, 7))

12
10
49
12 10 49


### `def`, вызов, параметры и аргументы

```python
def имя_функции(параметр1, параметр2):   # заголовок: def, имя, параметры в скобках, двоеточие
    тело                                 # с отступом; выполняется при каждом вызове
    return результат                     # что функция отдаёт наружу
```

Два термина, которые часто путают:

- **параметры** — имена в заголовке `def` (`width`, `height`): «что функция ожидает»;
- **аргументы** — конкретные значения при вызове (`3`, `4`): «что ей передали».

**Как Python выполняет `def`.** Строка `def` — это тоже инструкция. Дойдя до неё, интерпретатор
*создаёт объект-функцию* и привязывает его к имени; тело при этом **не выполняется**. Тело
запускается только при вызове `rectangle_area(3, 4)`: Python создаёт новое локальное пространство
имён, кладёт в него `width = 3` и `height = 4`, выполняет тело и на `return` возвращает значение
в точку вызова. После этого локальные переменные исчезают.

In [2]:
def greet(name):
    print("внутри функции: имя =", name)
    return "Привет, " + name + "!"

print("функция создана, но ещё не вызвана")
message = greet("Аня")        # вот здесь выполняется тело
print(message)
print(greet)                  # сама функция — тоже объект: у неё есть имя и адрес в памяти

функция создана, но ещё не вызвана
внутри функции: имя = Аня
Привет, Аня!
<function greet at 0x7fd96a542a20>


**Типичные ошибки**

- **Вызов до определения.** `def` должен выполниться раньше вызова — Python читает код сверху вниз.
- **Забыли скобки.** `greet` — объект-функция, `greet("Аня")` — её вызов. `x = greet` не вызывает
  функцию, а даёт ей второе имя.
- **Не то число аргументов.** `greet()` или `greet("Аня", "Боря")` →
  `TypeError: greet() takes 1 positional argument but 2 were given`. Сообщение всегда говорит,
  сколько ожидалось и сколько передано.

**Проверьте себя.** Что напечатает этот код?

```python
def f(x):
    return x * 2

print(f)
print(f(f(3)))
```

<details><summary>Ответ</summary>
Первая строка — что-то вроде <code>&lt;function f at 0x...&gt;</code>: печатается сам объект-функция.
Вторая — <code>12</code>: сначала <code>f(3)</code> даёт 6, потом <code>f(6)</code> даёт 12.
</details>

### `return` и `print` — разные вещи

`print` **показывает** значение на экране и больше ничего с ним не делает. `return` **отдаёт**
значение тому, кто вызвал функцию: его можно сохранить в переменную, передать в другую функцию,
сложить с чем-то. Функция без `return` (или с `return` без значения) возвращает `None`.

Ещё два свойства `return`:

- он **немедленно завершает** функцию — код после него не выполняется;
- вернуть несколько значений можно кортежем: `return total, average`, а принять — распаковкой
  `total, average = stats(nums)`.

In [3]:
def show_double(x):
    print(x * 2)            # только показывает

def double(x):
    return x * 2            # отдаёт значение

a = show_double(5)          # напечатает 10...
print("a =", a)             # ...но в a лежит None: функция ничего не вернула

b = double(5)               # ничего не печатает
print("b =", b, "| b + 1 =", b + 1)   # зато значение можно использовать дальше

def sign(x):
    if x < 0:
        return "отрицательное"
    return "неотрицательное"    # сюда попадаем, только если первый return не сработал

print(sign(-3), sign(7))

10
a = None
b = 10 | b + 1 = 11
отрицательное неотрицательное


**Типичные ошибки**

- `result = print(...)` — в `result` всегда `None`.
- Ветка без `return`: если написать только `if x < 0: return "отрицательное"`, то для `x = 7`
  функция вернёт `None`.
- `return` внутри цикла завершает всю функцию на первой же итерации — иногда это и нужно
  (поиск первого подходящего элемента), а иногда это ошибка.

#### Мини-задача 1. rectangle_area

**Условие.** Напишите функцию `rectangle_area(width, height)`, которая **возвращает** площадь прямоугольника. Печатать ничего не нужно: тест смотрит только на возвращаемое значение.

**Пример.**

```python
>>> rectangle_area(3, 4)
12
>>> rectangle_area(5, 5)
25
```

In [ ]:
from tests_functions_oop import test_rectangle_area

def rectangle_area(width, height):
    ...

test_rectangle_area(rectangle_area)

### Параметры по умолчанию

Параметру можно задать значение по умолчанию: `def power(base, exp=2)`. Тогда `power(3)` вернёт 9,
а `power(3, 3)` — 27. Правило одно: параметры со значениями по умолчанию идут **после** обычных.

**Как Python это выполняет.** Значение по умолчанию вычисляется **один раз**, в момент `def`,
и хранится вместе с функцией. Для чисел и строк это неважно, а для списков — ловушка:
`def add(item, items=[])` создаст один список на все вызовы, и он будет накапливать элементы
между вызовами. Правильно писать `items=None`, а внутри — `if items is None: items = []`.

In [5]:
def power(base, exp=2):
    return base ** exp

print(power(3), power(3, 3), power(2, exp=10))

# ловушка с изменяемым значением по умолчанию
def add_bad(item, items=[]):
    items.append(item)
    return items

print(add_bad(1))      # [1]
print(add_bad(2))      # [1, 2] — тот же список, что и в прошлый раз!

def add_good(item, items=None):
    if items is None:
        items = []     # новый список при каждом вызове без второго аргумента
    items.append(item)
    return items

print(add_good(1), add_good(2))

9 27 1024
[1]
[1, 2]
[1] [2]


### Позиционные и именованные аргументы: первое знакомство

Аргументы можно передавать **по позиции** — `describe("Аня", 20)`: первый к первому параметру,
второй ко второму — или **по имени** — `describe(age=20, name="Аня")`: порядок неважен.
Именованные аргументы делают вызов читаемым и позволяют менять только нужное значение
по умолчанию: `open("data.txt", encoding="utf-8")`.

Правила: сначала позиционные, потом именованные; один параметр нельзя задать дважды.
Подробнее — в разделе 4.2.

In [6]:
def describe(name, age, city="неизвестно"):
    return f"{name}, {age} лет, город: {city}"

print(describe("Аня", 20))
print(describe("Аня", 20, "Казань"))
print(describe(age=20, name="Аня"))           # порядок неважен
print(describe("Аня", city="Тверь", age=20))  # позиционные — до именованных

Аня, 20 лет, город: неизвестно
Аня, 20 лет, город: Казань
Аня, 20 лет, город: неизвестно
Аня, 20 лет, город: Тверь


**Проверьте себя.** Какой из вызовов вызовет ошибку и почему?

```python
describe("Аня", age=20)
describe("Аня", name="Боря", age=5)
```

<details><summary>Ответ</summary>
Второй: <code>TypeError: describe() got multiple values for argument 'name'</code> — параметр
<code>name</code> получил значение и по позиции («Аня»), и по имени.
</details>

### Области видимости: local, global, LEGB

Переменная, созданная внутри функции, — **локальная**: она существует, только пока выполняется
вызов, и не видна снаружи. Переменная, созданная на верхнем уровне ноутбука или файла, —
**глобальная**: её видно везде, в том числе внутри функций (для чтения).

Встретив имя, Python ищет его по правилу **LEGB** — в таком порядке:

1. **L**ocal — локальные имена текущей функции;
2. **E**nclosing — локальные имена внешних функций (если функция вложена в другую);
3. **G**lobal — имена модуля (ноутбука);
4. **B**uiltin — встроенные имена: `len`, `print`, `max`…

Используется первое найденное. Поэтому локальная `x` «заслоняет» глобальную `x`, а собственная
функция с именем `len` заслонила бы встроенную (не делайте так).

**Главная тонкость.** Присваивание внутри функции **создаёт локальную переменную** — даже если
глобальная с таким именем уже есть. Python решает это ещё до выполнения тела: если в функции есть
`x = ...`, то `x` считается локальной во *всей* функции. Отсюда `UnboundLocalError`:
`x += 1` — это тоже присваивание, а прочитать локальную `x` до присваивания нельзя.

In [7]:
x = 10                      # глобальная

def read_global():
    return x + 1            # чтение глобальной — можно

def shadow():
    x = 99                  # новая локальная x, глобальная не тронута
    return x

print(read_global(), shadow(), x)

def increment_bad():
    x += 1                  # x локальная (в функции есть присваивание), но ещё не создана → ошибка
    return x

try:
    increment_bad()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

def increment_global():
    global x                # явно говорим: работаем с глобальной x
    x += 1
    return x

print(increment_global(), x)

11 99 10
UnboundLocalError: cannot access local variable 'x' where it is not associated with a value
11 11


`global` работает, но это почти всегда плохой стиль: функция, которая молча меняет глобальные
переменные, непредсказуема. Правильный способ — принять значение как параметр и вернуть новое
через `return`.

Для вложенных функций есть `nonlocal`: он даёт доступ к переменной **внешней функции** (уровень E).
Так устроены **замыкания** — функции, которые «помнят» переменные из окружения, где были созданы.
Замыкания понадобятся в разделе 4.3 для декораторов.

In [8]:
def make_greeter(greeting):
    def greet(name):                 # greet видит greeting из внешней функции (уровень E)
        return f"{greeting}, {name}!"
    return greet

hello = make_greeter("Привет")
hi = make_greeter("Hi")
print(hello("Аня"), hi("Bob"))       # каждая функция помнит своё greeting

def make_accumulator():
    total = 0
    def add(value):
        nonlocal total               # меняем total внешней функции, а не создаём локальную
        total += value
        return total
    return add

acc = make_accumulator()
print(acc(10), acc(5), acc(1))

Привет, Аня! Hi, Bob!
10 15 16


**Типичные ошибки**

- `UnboundLocalError` при попытке «дописать» глобальную переменную внутри функции (`count += 1`).
- Одинаковые имена снаружи и внутри функции: код работает, но читать его тяжело — внутри
  функции это *другая* переменная.
- `nonlocal` для глобальной переменной — ошибка: он ищет имя только во внешних функциях.

**Проверьте себя.** Что напечатает код?

```python
n = 1

def f():
    n = 2
    def g():
        return n
    return g()

print(f(), n)
```

<details><summary>Ответ</summary>
<code>2 1</code>. Внутри <code>g</code> имя <code>n</code> находится на уровне E (в <code>f</code>),
поэтому 2; глобальная <code>n</code> не менялась.
</details>

#### Мини-задача 2. make_counter

**Условие.** Напишите функцию `make_counter()`, которая возвращает функцию-счётчик: каждый вызов счётчика возвращает 1, 2, 3, … Каждый созданный счётчик считает независимо от других. Используйте вложенную функцию и `nonlocal`.

**Пример.**

```python
>>> counter = make_counter()
>>> counter(), counter(), counter()
(1, 2, 3)
>>> other = make_counter()
>>> other()
1
```

In [ ]:
from tests_functions_oop import test_make_counter

def make_counter():
    count = 0

    def counter():
        ...

    return counter

test_make_counter(make_counter)

### Передача объектов в функцию: mutable и immutable

В Python аргументы передаются **по ссылке на объект**: функция получает не копию значения,
а тот же самый объект. Что из этого следует, зависит от того, изменяемый объект или нет:

- **неизменяемые** (`int`, `float`, `str`, `tuple`): изменить их нельзя в принципе, поэтому
  `x += 1` внутри функции создаёт *новый* объект и привязывает к локальному имени — снаружи
  ничего не меняется;
- **изменяемые** (`list`, `dict`, `set`): `items.append(...)` меняет *тот самый* объект —
  вызывающий код увидит изменения.

Есть и третий случай: `items = items + [1]`. Это не изменение списка, а создание нового
и привязка к локальному имени — снаружи список останется прежним.

In [10]:
def add_one(x):
    x += 1                  # новый int, локальное имя
    return x

def append_item(items):
    items.append(99)        # меняем сам список

def replace_list(items):
    items = items + [99]    # новый список, локальное имя
    return items

n = 5
lst = [1, 2]
add_one(n); append_item(lst); replace_list(lst)
print(n, lst)               # 5 [1, 2, 99]: сработал только append

def add_item_safely(items, item):
    result = items.copy()   # или list(items), или items[:]
    result.append(item)
    return result

original = [1, 2]
print(add_item_safely(original, 3), original)

5 [1, 2, 99]
[1, 2, 3] [1, 2]


**Типичные ошибки**

- Ожидать, что `x += 1` изменит число снаружи. Нужно вернуть значение: `n = add_one(n)`.
- Случайно изменить переданный список (`sort()`, `append`, `pop`) — вызывающий код этого не ждёт.
  Если функция должна вернуть «новый» список — делайте копию или собирайте новый список.
- Путать `sorted(lst)` (новый список) и `lst.sort()` (меняет на месте и возвращает `None`).

**Проверьте себя.** Что напечатает код?

```python
def f(d):
    d["a"] = 1
    d = {}

data = {}
f(data)
print(data)
```

<details><summary>Ответ</summary>
<code>{'a': 1}</code>. Первая строка изменила переданный словарь, а <code>d = {}</code> лишь
привязала локальное имя к новому пустому словарю.
</details>

#### Мини-задача 3. add_bonus

**Условие.** Напишите функцию `add_bonus(scores, bonus)`, которая возвращает **новый** список, где к каждому баллу прибавлен `bonus`. Исходный список должен остаться без изменений — тест это проверяет. Соберите результат списковым включением: методы `append`, `extend` и `insert` в этой мини-задаче запрещены.

**Пример.**

```python
>>> add_bonus([70, 75, 85], 10)
[80, 85, 95]
>>> add_bonus([], 5)
[]
```

In [ ]:
from tests_functions_oop import test_add_bonus

def add_bonus(scores, bonus):
    ...

test_add_bonus(add_bonus)

## 4.2. Позиционные и именованные аргументы. Функции высших порядков. Lambda-функции

### `*args`: произвольное число позиционных аргументов

`print` принимает сколько угодно аргументов: `print(1)`, `print(1, 2, 3)`. Так умеет любая функция,
если объявить параметр со звёздочкой: `def total(*numbers)`. Все «лишние» позиционные аргументы
Python **собирает в кортеж** `numbers`. Имя `args` — просто традиция, суть — в звёздочке.

Звёздочка работает и в другую сторону, при **вызове**: `total(*[1, 2, 3])` **распаковывает** список
в отдельные аргументы. Это удобно, когда данные уже лежат в списке или кортеже.

In [12]:
def total(*numbers):
    print("numbers =", numbers)      # это кортеж
    return sum(numbers)

print(total())
print(total(1, 2, 3))

values = [10, 20, 30]
print(total(*values))                # распаковка: total(10, 20, 30)

try:
    print(total(values))             # без звёздочки: numbers == ([10, 20, 30],) — один аргумент-список
except TypeError as e:
    print("TypeError:", e)

numbers = ()
0
numbers = (1, 2, 3)
6
numbers = (10, 20, 30)
60
numbers = ([10, 20, 30],)
TypeError: unsupported operand type(s) for +: 'int' and 'list'


### `**kwargs` и порядок параметров

Аналогично `**kwargs` собирает все «лишние» **именованные** аргументы в словарь:
`describe(name="Аня", age=20)` → `{"name": "Аня", "age": 20}`. И снова две звёздочки при вызове
распаковывают словарь: `describe(**data)`.

Полный порядок параметров в заголовке функции:

```
def f(обычные, *args, только_именованные, **kwargs)
```

Всё, что стоит после `*args` (или после одиночной `*`), можно передать **только по имени** — так
делают параметры-настройки, которые опасно перепутать местами.

**Как Python сопоставляет аргументы.** Сначала позиционные аргументы раздаются обычным параметрам
слева направо, остаток уходит в `*args`. Потом именованные: если есть параметр с таким именем —
ему, иначе в `**kwargs`. Если параметру без значения по умолчанию ничего не досталось —
`TypeError`.

In [13]:
def describe(**fields):
    return ", ".join(f"{key}={value}" for key, value in fields.items())

print(describe(name="Аня", age=20))
data = {"city": "Казань", "year": 1}
print(describe(**data))

def report(title, *values, sep=" | ", **options):
    line = sep.join(str(v) for v in values)
    return f"{title}: {line} {options}"

print(report("Оценки", 5, 4, 5))
print(report("Оценки", 5, 4, 5, sep=", ", bold=True))

def connect(host, *, port=80, timeout=5):     # port и timeout — только по имени
    return f"{host}:{port} (timeout {timeout})"

print(connect("example.com", port=8080))
try:
    connect("example.com", 8080)
except TypeError as e:
    print("TypeError:", e)

name=Аня, age=20
city=Казань, year=1
Оценки: 5 | 4 | 5 {}
Оценки: 5, 4, 5 {'bold': True}
example.com:8080 (timeout 5)
TypeError: connect() takes 1 positional argument but 2 were given


**Типичные ошибки**

- Передать список туда, где ждут отдельные аргументы (без `*`), или наоборот.
- Имя в `**kwargs` совпало с обычным параметром: при `def f(x, **kw)` вызов `f(x=1)` кладёт
  значение в `x`, а не в `kw`.
- Порядок в заголовке: `def f(**kwargs, *args)` — синтаксическая ошибка.

**Проверьте себя.** Что окажется в `x`, `args` и `kwargs` при вызове `f(1, 2, 3, a=4)`
для `def f(x, *args, **kwargs)`?

<details><summary>Ответ</summary>
<code>x = 1</code>, <code>args = (2, 3)</code>, <code>kwargs = {'a': 4}</code>.
</details>

#### Мини-задача 4. join_words

**Условие.** Напишите функцию `join_words(sep, *words)`, которая соединяет все переданные слова через разделитель `sep`. Если слов не передали — вернуть пустую строку.

**Пример.**

```python
>>> join_words('-', 'a', 'b', 'c')
'a-b-c'
>>> join_words(' ', 'hello', 'world')
'hello world'
>>> join_words(', ')
''
```

In [ ]:
from tests_functions_oop import test_join_words

def join_words(sep, *words):
    ...

test_join_words(join_words)

### Функции — это объекты

В Python функция — обычный объект, как число или список. Её можно положить в переменную, в список
или словарь, передать в другую функцию как аргумент и вернуть из функции. У неё есть атрибуты:
`__name__`, `__doc__`.

Это открывает главный приём раздела — **функции высших порядков**: функции, которые принимают
другие функции или возвращают их. Одну вы уже видели: `make_greeter` возвращала функцию.
А `sorted(..., key=...)` — принимает.

In [15]:
def shout(text):
    return text.upper() + "!"

def whisper(text):
    return text.lower() + "..."

say = shout                       # второе имя для той же функции
print(say("привет"), say.__name__)

styles = {"громко": shout, "тихо": whisper}       # функции в словаре
print(styles["тихо"]("ПРИВЕТ"))

def apply_twice(func, value):     # принимает функцию
    return func(func(value))

print(apply_twice(shout, "эй"))

def make_multiplier(n):           # возвращает функцию
    def multiply(x):
        return x * n
    return multiply

triple = make_multiplier(3)
print(triple(7), make_multiplier(10)(7))

ПРИВЕТ! shout
привет...
ЭЙ!!
21 70


**Проверьте себя.** Чем отличаются `apply_twice(shout, "эй")` и `apply_twice(shout("эй"))`?

<details><summary>Ответ</summary>
В первом случае передаётся сама функция <code>shout</code>. Во втором сначала вычисляется
<code>shout("эй")</code> — это строка, и <code>apply_twice</code> получает один аргумент вместо
двух → <code>TypeError</code>.
</details>

#### Мини-задача 5. compose

**Условие.** Напишите функцию `compose(f, g)`, которая возвращает новую функцию `h`, такую что `h(x) == f(g(x))`: сначала применяется `g`, потом `f`.

**Пример.**

```python
>>> to_len = compose(len, str)
>>> to_len(12345)
5
>>> compose(lambda x: x + 1, lambda x: x * 2)(3)
7
```

In [ ]:
from tests_functions_oop import test_compose

def compose(f, g):
    ...

test_compose(compose)

### lambda-функции

`lambda` — способ записать маленькую функцию из одного выражения прямо там, где она нужна, без
`def` и имени:

```python
lambda параметры: выражение
```

`lambda x: x * 2` — то же, что `def double(x): return x * 2`, только без имени. Тело — ровно одно
выражение, его значение и возвращается; `return`, циклы и инструкцию `if` внутри писать нельзя
(условное *выражение* `a if c else b` — можно).

Используйте `lambda` там, где функция нужна один раз и она короткая — обычно как аргумент `key=`
или для `map`/`filter`. Если логика сложнее одной строки — пишите `def`: имя и docstring важнее
краткости.

In [17]:
double = lambda x: x * 2          # так делать не стоит — для именованной функции есть def
print(double(4))

people = [("Аня", 22), ("Боря", 19), ("Вера", 31)]
print(sorted(people, key=lambda person: person[1]))          # по возрасту
print(max(people, key=lambda person: len(person[0])))         # самое длинное имя
print(sorted(people, key=lambda p: (-p[1], p[0])))            # по убыванию возраста, при равенстве — по имени

8
[('Боря', 19), ('Аня', 22), ('Вера', 31)]
('Боря', 19)
[('Вера', 31), ('Аня', 22), ('Боря', 19)]


### `map`, `filter`, `sorted` с `key`

Три встроенные функции высшего порядка, которые чаще всего получают `lambda`:

- `map(func, iterable)` — применяет `func` к каждому элементу;
- `filter(func, iterable)` — оставляет элементы, для которых `func` вернула истину;
- `sorted(iterable, key=func, reverse=...)` — сортирует по значению `func`.

`map` и `filter` возвращают не список, а **ленивый итератор**: элементы вычисляются по мере обхода.
Чтобы увидеть результат целиком — `list(...)`; чтобы просуммировать — `sum(...)`; в цикле `for`
итератор можно использовать напрямую. Пройти по итератору можно только один раз.

Списковое включение `[x * 2 for x in nums if x > 0]` делает то же, что `map` + `filter`, и часто
читается лучше. Но `map`/`filter` удобны, когда функция уже есть: `map(int, strings)`,
`filter(str.isdigit, tokens)`.

In [18]:
nums = [3, -1, 4, -1, 5, -9]

squares = map(lambda x: x * x, nums)
print(squares)                    # итератор, а не список
print(list(squares))
print(list(squares))              # второй раз — пусто: итератор исчерпан

positives = list(filter(lambda x: x > 0, nums))
print(positives)

print(list(map(int, ["1", "2", "3"])))          # готовая функция вместо lambda
print(sorted(nums, key=abs))                     # по модулю
print(sorted(["bb", "a", "ccc"], key=len, reverse=True))
print(sum(map(lambda x: x * x, filter(lambda x: x > 0, nums))))   # конвейер

[9, 1, 16, 1, 25, 81]
[]
[3, 4, 5]
[1, 2, 3]
[-1, -1, 3, 4, 5, -9]
['ccc', 'bb', 'a']
50


**Типичные ошибки**

- Напечатать `map(...)` и удивиться `<map object at ...>` — оберните в `list`.
- Использовать итератор дважды — второй проход пустой.
- `sorted(lst, key=len())` — вызвали `len` вместо того, чтобы передать её; нужно `key=len`.

**Проверьте себя.** Что напечатает
`list(map(lambda x: x + 1, filter(lambda x: x % 2, [1, 2, 3, 4])))`?

<details><summary>Ответ</summary>
<code>[2, 4]</code>: <code>filter</code> оставил нечётные 1 и 3 (остаток 1 — истина),
<code>map</code> прибавил единицу.
</details>

#### Мини-задача 6. sort_by_length

**Условие.** Напишите функцию `sort_by_length(words)`, которая возвращает новый список слов, отсортированный по длине; слова одинаковой длины — по алфавиту. Используйте `sorted` с `key=lambda ...` и кортежем в качестве ключа. Метод `list.sort()` не используйте: он изменил бы исходный список.

**Пример.**

```python
>>> sort_by_length(['bb', 'a', 'ccc'])
['a', 'bb', 'ccc']
>>> sort_by_length(['ab', 'aa', 'b'])
['b', 'aa', 'ab']
```

In [ ]:
from tests_functions_oop import test_sort_by_length

def sort_by_length(words):
    ...

test_sort_by_length(sort_by_length)

## 4.3. Рекурсия. Декораторы. Генераторы

### Рекурсия

Рекурсивная функция — та, что вызывает сама себя. Это естественный способ решать задачи, которые
сводятся к **той же задаче меньшего размера**: факториал `n! = n · (n−1)!`; сумма вложенного
списка — сумма его элементов, где вложенные списки считаются тем же способом.

У любой рекурсии две части:

- **базовый случай** — самая маленькая задача, ответ на которую известен без рекурсии
  (`0! = 1`, сумма пустого списка — 0);
- **рекурсивный шаг** — свести задачу к меньшей и вызвать себя.

**Как Python выполняет рекурсию.** Каждый вызов функции получает свой кадр в **стеке вызовов** —
своё пространство локальных переменных. `factorial(3)` вызывает `factorial(2)`, тот —
`factorial(1)`, и только когда самый глубокий вызов вернул значение, кадры начинают
«сворачиваться» снизу вверх, перемножая результаты. Глубина стека ограничена (около 1000 кадров
по умолчанию): без базового случая или с недостижимым базовым случаем получите
`RecursionError: maximum recursion depth exceeded`.

In [20]:
def factorial(n, depth=0):
    indent = "  " * depth
    print(f"{indent}factorial({n}) вызвана")
    if n == 0:                                   # базовый случай
        print(f"{indent}базовый случай → 1")
        return 1
    result = n * factorial(n - 1, depth + 1)     # рекурсивный шаг
    print(f"{indent}factorial({n}) возвращает {result}")
    return result

print(factorial(3))

def nested_sum(items):
    total = 0
    for item in items:
        if isinstance(item, list):
            total += nested_sum(item)            # список внутри — та же задача меньшего размера
        else:
            total += item
    return total

print(nested_sum([1, [2, 3, [4]], 5]))

factorial(3) вызвана
  factorial(2) вызвана
    factorial(1) вызвана
      factorial(0) вызвана
      базовый случай → 1
    factorial(1) возвращает 1
  factorial(2) возвращает 2
factorial(3) возвращает 6
6
15


**Типичные ошибки**

- Нет базового случая или он недостижим (`factorial(n - 1)` при отрицательном `n`) →
  `RecursionError`.
- Забыли `return` перед рекурсивным вызовом: `factorial(n - 1)` вычисляется и выбрасывается,
  функция возвращает `None`.
- Рекурсия там, где хватит цикла: для перебора списка цикл проще и быстрее; рекурсия хороша для
  вложенных структур — деревьев, вложенных списков, папок с подпапками.

**Проверьте себя.** Что вернёт `f(3)`?

```python
def f(n):
    if n == 0:
        return 0
    return n + f(n - 1)
```

<details><summary>Ответ</summary>
<code>6</code>: 3 + 2 + 1 + 0 — рекурсивная сумма чисел от 0 до n.
</details>

#### Мини-задача 7. sum_digits

**Условие.** Напишите рекурсивную функцию `sum_digits(n)`, возвращающую сумму цифр неотрицательного целого `n`. Базовый случай — однозначное число; рекурсивный шаг — последняя цифра (`n % 10`) плюс сумма цифр остального числа (`n // 10`). Проверка убедится, что функция вызывает сама себя.

**Пример.**

```python
>>> sum_digits(123)
6
>>> sum_digits(0)
0
>>> sum_digits(9)
9
```

In [ ]:
from tests_functions_oop import test_sum_digits

def sum_digits(n):
    ...

test_sum_digits(sum_digits)

### Декораторы

Декоратор — это функция, которая принимает функцию и возвращает **новую функцию**, обычно
расширяющую поведение исходной: замерить время, записать вызов в лог, проверить аргументы,
закэшировать результат. Саму исходную функцию при этом менять не нужно.

Запись

```python
@timer
def slow():
    ...
```

— это ровно то же самое, что

```python
def slow():
    ...
slow = timer(slow)
```

`@` — синтаксический сахар: сразу после `def` Python вызывает `timer(slow)` и привязывает к имени
`slow` то, что вернул декоратор.

Внутри декоратора обычно живёт **обёртка** (`wrapper`): она принимает любые аргументы
(`*args, **kwargs`), делает что-то до, вызывает исходную функцию, делает что-то после — и
**возвращает её результат**. Исходная функция доступна обёртке как переменная внешней функции —
это замыкание из раздела 4.1.

In [22]:
import time
from functools import wraps

def timer(func):
    @wraps(func)                        # сохраняет имя и docstring исходной функции
    def wrapper(*args, **kwargs):
        started = time.perf_counter()
        result = func(*args, **kwargs)  # вызываем исходную функцию
        elapsed = time.perf_counter() - started
        print(f"{func.__name__} заняла {elapsed:.4f} с")
        return result                   # не забываем вернуть результат
    return wrapper

@timer
def count_to(n):
    """Считает сумму чисел до n."""
    total = 0
    for i in range(n):
        total += i
    return total

print(count_to(100_000))
print(count_to.__name__, "|", count_to.__doc__)   # благодаря wraps — count_to, а не wrapper

count_to заняла 0.0019 с
4999950000
count_to | Считает сумму чисел до n.


**Зачем `functools.wraps`.** Без него `count_to.__name__` было бы `"wrapper"`, а docstring
пропал бы: отладчики и `help()` показывали бы обёртку вместо вашей функции. `@wraps(func)` копирует
имя и docstring и кладёт ссылку на оригинал в атрибут `__wrapped__`.

**Декоратор с параметрами** — это функция, которая *возвращает декоратор*: `@repeat(3)` сначала
вызывает `repeat(3)`, получает декоратор, и уже он применяется к функции. Три уровня вложенности
выглядят пугающе, но логика та же.

In [23]:
from functools import wraps

def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def hello(name):
    print("привет,", name)

hello("Аня")

привет, Аня
привет, Аня
привет, Аня


**Типичные ошибки**

- Обёртка не возвращает результат `func(...)` — декорированная функция всегда возвращает `None`.
- Декоратор возвращает не `wrapper`, а результат `func()`, то есть вызывает функцию на этапе
  декорирования.
- `wrapper()` без `*args, **kwargs` — декоратор работает только для функций без параметров.
- Забыли `@wraps` — имя функции стало `wrapper`.

**Проверьте себя.** Что напечатает код?

```python
def deco(func):
    print("декорирую", func.__name__)
    return func

@deco
def f():
    print("вызов f")
```

<details><summary>Ответ</summary>
Только <code>декорирую f</code>: декоратор выполняется в момент определения, а <code>f()</code>
никто не вызывал. Сам <code>deco</code> вернул функцию без изменений.
</details>

#### Мини-задача 8. double_result

**Условие.** Напишите декоратор `double_result(func)`. Его обёртка вызывает `func` с теми же аргументами (позиционными и именованными) и возвращает результат, умноженный на 2. Используйте `functools.wraps`, чтобы сохранить имя исходной функции.

**Пример.**

```python
>>> @double_result
... def add(a, b):
...     return a + b
>>> add(2, 3)
10
>>> add.__name__
'add'
```

In [ ]:
from tests_functions_oop import test_double_result

from functools import wraps

def double_result(func):
    ...

test_double_result(double_result)

### Генераторы

Функция с `yield` вместо `return` — **генератор**. При вызове она не выполняется, а возвращает
объект-генератор; тело запускается, только когда у генератора просят следующее значение
(`next()` или цикл `for`). Дойдя до `yield`, функция **отдаёт значение и замирает**, сохраняя все
локальные переменные; при следующем запросе продолжает с того же места. Когда тело заканчивается,
генератор сообщает об этом исключением `StopIteration`, и цикл `for` завершается.

Зачем это нужно:

- **память** — длинные последовательности не нужно хранить целиком, значения появляются по одному;
- **бесконечные последовательности** — генератор может никогда не заканчиваться, а потребитель
  возьмёт столько, сколько нужно;
- **конвейеры** — один генератор читает данные, другой фильтрует, третий преобразует, и ничего
  не вычисляется раньше времени.

`yield` против `return`: `return` завершает функцию и отдаёт одно значение; `yield` отдаёт значение
и *приостанавливает* функцию, значений может быть много. `return` внутри генератора просто
завершает его.

In [25]:
def countdown(n):
    print("старт")
    while n > 0:
        yield n            # отдаём значение и замираем
        n -= 1
    print("финиш")

gen = countdown(3)
print(gen)                 # объект-генератор; тело ещё не выполнялось
print(next(gen))           # "старт", затем 3
print(next(gen))           # 2
for value in gen:          # остаток: 1, потом "финиш"
    print("в цикле:", value)

def fibonacci():           # бесконечный генератор
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
print([next(fib) for _ in range(10)])

<generator object countdown at 0x7fd9698ff1d0>
старт
3
2
в цикле: 1
финиш
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


**Генераторные выражения** — то же, что списковые включения, но в круглых скобках и ленивые:
`(x * x for x in range(10))`. Их удобно передавать прямо в `sum`, `max`, `any`, `" ".join` — без
промежуточного списка. `sum(x * x for x in range(10**6))` считает сумму, не занимая память под
миллион чисел.

In [26]:
squares_list = [x * x for x in range(5)]       # список: все значения сразу в памяти
squares_gen = (x * x for x in range(5))        # генератор: по одному

print(squares_list, squares_gen)
print(sum(squares_gen))
print(sum(squares_gen))                        # 0: генератор уже исчерпан
print(any(word.startswith("п") for word in ["кот", "пёс"]))

[0, 1, 4, 9, 16] <generator object <genexpr> at 0x7fd9698fd560>
30
0
True


**Типичные ошибки**

- Напечатать генератор и ждать значений — увидите `<generator object ...>`; используйте `list(...)`
  или цикл.
- Пройти по генератору дважды: второй раз он пуст. Нужны два прохода — создайте генератор заново
  или сохраните `list`.
- Функция с `yield` не выполняется при вызове: `print` в её начале не сработает, пока не запросят
  первое значение.

**Проверьте себя.** Что напечатает код?

```python
def g():
    yield 1
    return 2
    yield 3

print(list(g()))
```

<details><summary>Ответ</summary>
<code>[1]</code>. <code>return</code> завершает генератор; до второго <code>yield</code> дело не доходит.
</details>

#### Мини-задача 9. evens_up_to

**Условие.** Напишите функцию-генератор `evens_up_to(n)`, которая по очереди отдаёт чётные числа от 0 до `n` включительно. Для отрицательного `n` генератор не отдаёт ничего. Проверка убедится, что это именно генератор (функция с `yield`), а не список.

**Пример.**

```python
>>> list(evens_up_to(10))
[0, 2, 4, 6, 8, 10]
>>> next(evens_up_to(10))
0
>>> list(evens_up_to(-1))
[]
```

In [ ]:
from tests_functions_oop import test_evens_up_to

def evens_up_to(n):
    ...

test_evens_up_to(evens_up_to)

### Итоги главы 4

| Тема | Главное |
|---|---|
| `def` / `return` | тело выполняется при вызове; `return` отдаёт значение и завершает функцию; без `return` — `None` |
| Области видимости | LEGB; присваивание создаёт локальную переменную; `global`/`nonlocal` — явный доступ наружу |
| Передача объектов | по ссылке на объект: изменяемые можно изменить внутри, неизменяемые — нет |
| `*args` / `**kwargs` | собирают лишние аргументы в кортеж/словарь; `*`/`**` при вызове распаковывают |
| Функции высших порядков | функции — объекты: их передают, возвращают, хранят в словарях |
| `lambda` | безымянная функция из одного выражения — для `key=`, `map`, `filter` |
| Рекурсия | базовый случай + рекурсивный шаг; стек вызовов ограничен |
| Декораторы | `@d` ≡ `f = d(f)`; обёртка с `*args, **kwargs`; `functools.wraps` |
| Генераторы | `yield` отдаёт значение и замирает; ленивые, одноразовые; `(x for x in ...)` |

## 4.4. Десять заданий на функции

Задания идут от простого к сложному, и каждое проверяет отдельный навык. Для каждого:
**Условие**, **Что проверяется**, **Ограничения** (если есть), **Примеры**, шаблон и автопроверка.
Решения — в конце ноутбука, в разделе «Разбор решений», но сначала попробуйте сами: смысл именно
в том, чтобы написать код самостоятельно и увидеть, на каком тесте он ломается.

### Задача 1. clamp — число в границах

**Условие.** Напишите функцию `clamp(value, low, high)`, которая возвращает `value`, «прижатое» к отрезку `[low, high]`: если `value` меньше `low` — вернуть `low`; если больше `high` — вернуть `high`; иначе — само `value`. Гарантируется, что `low <= high`. Числа могут быть целыми и дробными, в том числе отрицательными.

**Что проверяется.** функция с несколькими параметрами; `return` вместо `print`; условия или связка `min`/`max`.

**Ограничения.** результат нужно **вернуть** через `return`; функция ничего не печатает (проверяется).

**Пример.**

```python
>>> clamp(5, 0, 10)
5
>>> clamp(-3, 0, 10)
0
>>> clamp(42, 0, 10)
10
```

In [ ]:
from tests_functions_oop import test_clamp

def clamp(value, low, high):
    ...

test_clamp(clamp)

### Задача 2. snake_to_camel — работа со строками

**Условие.** Имена в стиле snake_case (`make_http_request`) нужно переводить в camelCase (`makeHttpRequest`). Напишите функцию `snake_to_camel(text)`: слова разделены символом `_`; первое слово остаётся в нижнем регистре, каждое следующее пишется с заглавной буквы, остальные буквы — строчные. Входная строка может быть в любом регистре: `HELLO_WORLD` → `helloWorld`. Строка без `_` просто приводится к нижнему регистру.

**Что проверяется.** методы строк `lower`, `split`, `capitalize`, `join`; срезы списка; сборка строки из частей.

**Ограничения.** используйте `return`.

**Пример.**

```python
>>> snake_to_camel('hello_world')
'helloWorld'
>>> snake_to_camel('x')
'x'
>>> snake_to_camel('a_b_c')
'aBC'
>>> snake_to_camel('make_http_request')
'makeHttpRequest'
```

In [ ]:
from tests_functions_oop import test_snake_to_camel

def snake_to_camel(text):
    ...

test_snake_to_camel(snake_to_camel)

### Задача 3. chunk — работа со списками

**Условие.** Напишите функцию `chunk(items, size)`, которая разбивает список `items` на подряд идущие куски по `size` элементов и возвращает список этих кусков. Последний кусок может быть короче. Для пустого списка вернуть `[]`. `size` — целое число не меньше 1.

**Что проверяется.** срезы списков; `range` с шагом; построение нового списка.

**Ограничения.** результат — **список списков** (не кортежей); исходный список не изменяется (проверяется).

**Пример.**

```python
>>> chunk([1, 2, 3, 4, 5], 2)
[[1, 2], [3, 4], [5]]
>>> chunk([], 3)
[]
>>> chunk([1, 2, 3], 3)
[[1, 2, 3]]
>>> chunk([1, 2, 3], 5)
[[1, 2, 3]]
```

In [ ]:
from tests_functions_oop import test_chunk

def chunk(items, size):
    ...

test_chunk(chunk)

### Задача 4. make_email — значения по умолчанию и именованные аргументы

**Условие.** Напишите функцию `make_email(first, last, domain="example.com", separator=".")`, которая собирает адрес электронной почты: имя и фамилия в нижнем регистре, между ними `separator`, затем `@` и `domain`. Параметры `domain` и `separator` должны иметь указанные значения по умолчанию, чтобы их можно было опускать или передавать по имени в любом порядке.

**Что проверяется.** параметры по умолчанию; вызов с позиционными и именованными аргументами; f-строки.

**Ограничения.** сигнатура проверяется: у `domain` значение по умолчанию `"example.com"`, у `separator` — `"."`.

**Пример.**

```python
>>> make_email('Ivan', 'Petrov')
'ivan.petrov@example.com'
>>> make_email('Anna', 'Smith', domain='mail.ru')
'anna.smith@mail.ru'
>>> make_email('Anna', 'Smith', separator='_')
'anna_smith@example.com'
>>> make_email(separator='-', last='Lee', first='Bo', domain='x.io')
'bo-lee@x.io'
```

In [ ]:
from tests_functions_oop import test_make_email

def make_email(first, last, domain="example.com", separator="."):
    ...

test_make_email(make_email)

### Задача 5. average — *args

**Условие.** Напишите функцию `average(*nums)`, которая принимает произвольное число чисел и возвращает их среднее арифметическое как `float`. Если чисел не передали — вернуть `0.0`. Функция должна работать и при распаковке списка: `average(*[4, 5, 6])`.

**Что проверяется.** параметр `*args`; работа с кортежем аргументов; деление и `len`.

**Ограничения.** аргументы принимаются через `*` (проверяется по сигнатуре), а не списком.

**Пример.**

```python
>>> average(1, 2, 3)
2.0
>>> average(10)
10.0
>>> average()
0.0
>>> average(1, 2)
1.5
```

In [ ]:
from tests_functions_oop import test_average

def average(*nums):
    ...

test_average(average)

### Задача 6. make_tag — **kwargs

**Условие.** Напишите функцию `make_tag(name, **attrs)`, которая строит открывающий HTML-тег: имя тега, затем атрибуты в виде `ключ="значение"` через пробел, в том порядке, в каком они переданы. Значения подставляются через `str` без изменений: число `100` → `"100"`, пустая строка → `""`. Без атрибутов результат — `<name>`.

**Что проверяется.** параметр `**kwargs`; обход словаря через `.items()`; сборка строки через `join`.

**Ограничения.** атрибуты принимаются через `**` (проверяется по сигнатуре).

**Пример.**

```python
>>> make_tag('br')
'<br>'
>>> make_tag('a', href='x', target='_blank')
'<a href="x" target="_blank">'
>>> make_tag('img', src='cat.png', width=100)
'<img src="cat.png" width="100">'
```

In [ ]:
from tests_functions_oop import test_make_tag

def make_tag(name, **attrs):
    ...

test_make_tag(make_tag)

### Задача 7. apply_pipeline — функция высшего порядка

**Условие.** Напишите функцию `apply_pipeline(value, *funcs)`, которая применяет к `value` функции из `funcs` **по очереди**: результат первой передаётся во вторую и так далее, и возвращает итоговое значение. Если функций не передали — вернуть `value` как есть. Функции могут быть любыми: встроенными (`str`, `len`, `abs`), методами (`str.strip`), `lambda`.

**Что проверяется.** функции как аргументы (функция высшего порядка); `*args`; цикл с накоплением результата.

**Ограничения.** функции принимаются через `*funcs` (проверяется по сигнатуре).

**Пример.**

```python
>>> apply_pipeline(3, lambda x: x + 1, lambda x: x * 2)
8
>>> apply_pipeline("  hi ", str.strip, str.upper)
'HI'
>>> apply_pipeline(5)
5
```

In [ ]:
from tests_functions_oop import test_apply_pipeline

def apply_pipeline(value, *funcs):
    ...

test_apply_pipeline(apply_pipeline)

### Задача 8. even_squares_desc — lambda + filter/map/sorted

**Условие.** Напишите функцию `even_squares_desc(nums)`, которая возвращает список квадратов **чётных** чисел из `nums`, отсортированный по убыванию. Ноль — чётное число; отрицательные числа тоже бывают чётными.

**Что проверяется.** `filter` и `map` с `lambda`; `sorted` с `reverse=True`; понимание, что `map` и `filter` возвращают итераторы.

**Ограничения.** обязательно использовать `filter`, `map`, `sorted` и хотя бы одну `lambda`; списковые включения в этой задаче запрещены (проверяется по коду).

**Пример.**

```python
>>> even_squares_desc([1, 2, 3, 4, 5, 6])
[36, 16, 4]
>>> even_squares_desc([])
[]
>>> even_squares_desc([1, 3, 5])
[]
```

In [ ]:
from tests_functions_oop import test_even_squares_desc

def even_squares_desc(nums):
    ...

test_even_squares_desc(even_squares_desc)

### Задача 9. flatten — рекурсия

**Условие.** Напишите рекурсивную функцию `flatten(nested)`, которая «разглаживает» список произвольной вложенности в плоский список элементов, сохраняя порядок. Вложенным считается любой элемент типа `list`; всё остальное — обычные элементы (числа, строки).

**Что проверяется.** рекурсия: базовый случай (элемент — не список) и рекурсивный шаг (элемент — список); `isinstance`; `extend` и `append`.

**Ограничения.** функция должна вызывать сама себя (проверяется); исходный список не изменяется.

**Пример.**

```python
>>> flatten([1, [2, [3, [4]]]])
[1, 2, 3, 4]
>>> flatten([])
[]
>>> flatten([[], [[]], 1])
[1]
```

In [ ]:
from tests_functions_oop import test_flatten

def flatten(nested):
    ...

test_flatten(flatten)

### Задача 10. memoize + fib — декоратор-кэш и рекурсия

**Условие.** Наивная рекурсивная функция Фибоначчи `fib(n) = fib(n-1) + fib(n-2)` считает одни и те же значения тысячи раз: `fib(35)` — секунды, `fib(90)` — не дождаться. Исправьте это декоратором.

1. Напишите декоратор `memoize(func)`, который запоминает результаты вызовов: обёртка хранит словарь `{аргументы: результат}`; при повторном вызове с теми же позиционными аргументами возвращает сохранённое значение, не вызывая `func`. Ключом словаря может быть кортеж `args`. Используйте `functools.wraps`.
2. Напишите рекурсивную функцию `fib(n)` (`fib(0) = 0`, `fib(1) = 1`) и примените к ней `@memoize`.

**Что проверяется.** декоратор с замыканием и `*args`; словарь как кэш; `functools.wraps`; рекурсия; как кэш превращает экспоненциальное число вызовов в линейное.

**Ограничения.** `memoize` использует `wraps`; `fib` рекурсивна и обёрнута `@memoize` (проверяется). Считайте, что функции вызываются только с позиционными аргументами.

**Пример.**

```python
>>> @memoize
... def slow_square(x):
...     print("считаю", x)
...     return x * x
>>> slow_square(4)
считаю 4
16
>>> slow_square(4)      # второй раз — из кэша, без печати
16
>>> fib(10), fib(90)
(55, 2880067194370816120)
```

In [ ]:
from tests_functions_oop import test_memoize_fib

from functools import wraps

def memoize(func):
    cache = {}
    ...


@memoize
def fib(n):
    ...

test_memoize_fib(memoize, fib)

# Глава 5. ООП и другие возможности Python

## 5.1. Объектная модель Python. Классы, поля и методы

### Объекты и классы

Всё в Python — объект: у числа, строки, списка и даже функции есть тип (`type(x)`), атрибуты и
методы. **Класс** — «чертёж» объектов определённого типа: он описывает, какие данные хранит каждый
объект (поля, они же атрибуты) и что объект умеет (методы). **Экземпляр** — конкретный объект,
созданный по чертежу. `list` — класс, `[1, 2]` — его экземпляр; `"abc".upper()` — вызов метода
класса `str`.

Зачем ООП: данные и функции для работы с ними хранятся вместе. Без классов «точка» — это кортеж
`(3, 4)` плюс отдельные функции `move(point, dx, dy)`, и нужно помнить, что с чем сочетается.
С классом — `point.move(1, 2)`: объект сам знает, что с ним можно делать.

In [38]:
print(type(42), type("abc"), type([1, 2]), type(print))
print(isinstance(42, int), isinstance("abc", int))
print("abc".upper(), [3, 1, 2].count(1))     # методы: функции, привязанные к объекту

<class 'int'> <class 'str'> <class 'list'> <class 'builtin_function_or_method'>
True False
ABC 1


### `class`, `__init__`, `self`

```python
class Point:
    def __init__(self, x, y):   # конструктор: вызывается при создании объекта
        self.x = x              # поле экземпляра
        self.y = y

    def move(self, dx, dy):     # метод: функция внутри класса, первый параметр — self
        self.x += dx
        self.y += dy

p = Point(1, 2)                 # создание экземпляра
p.move(3, 4)                    # вызов метода
```

**Как Python это выполняет.** `Point(1, 2)`: (1) создаётся пустой объект типа `Point`; (2) вызывается
`Point.__init__(объект, 1, 2)` — `self` и есть этот объект; (3) `self.x = x` создаёт атрибут у *этого*
объекта; (4) объект возвращается и привязывается к имени `p`. Вызов `p.move(3, 4)` — это
`Point.move(p, 3, 4)`: Python сам подставляет объект первым аргументом. Поэтому `self` пишется
в `def`, но не пишется при вызове.

`self` — не ключевое слово, а соглашение об имени первого параметра. Но следовать ему нужно всегда:
любой читающий код ожидает именно `self`.

In [39]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def move(self, dx, dy):
        self.x += dx
        self.y += dy

    def distance_to_origin(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

p = Point(3, 4)
print(p.x, p.y, p.distance_to_origin())
p.move(1, 1)
print(p.x, p.y)
Point.move(p, -1, -1)                  # то же самое «вручную»: метод — это функция класса
print(p.x, p.y)
q = Point(0, 0)
print(q.x, q.y, p is q)                # у каждого экземпляра свои поля

3 4 5.0
4 5
3 4
0 0 False


**Типичные ошибки**

- Забыли `self` в `def`: `def move(dx, dy)` →
  `TypeError: move() takes 2 positional arguments but 3 were given` — Python передал объект первым
  аргументом, а места для него нет.
- Забыли `self.` при присваивании: `x = x` создаёт локальную переменную, поле объекта не появляется →
  `AttributeError: 'Point' object has no attribute 'x'`.
- Вызов метода без скобок: `p.move` — сам метод, а не результат его вызова.
- `__init__` с `return значение` → `TypeError`: конструктор ничего не возвращает.

**Проверьте себя.** Сколько объектов `Point` будет создано и что напечатает код?

```python
a = Point(1, 1)
b = a
b.move(1, 1)
print(a.x, a.y)
```

<details><summary>Ответ</summary>
Один объект: <code>b = a</code> — это второе имя того же объекта, а не копия. Напечатает
<code>2 2</code>.
</details>

### Атрибуты экземпляра и атрибуты класса

Поля, созданные через `self.x = ...`, принадлежат экземпляру: у каждого объекта своя копия.
Переменная, объявленная прямо в теле класса, — **атрибут класса**: он один на всех и виден
и через `Counter.created`, и через любой экземпляр. Так обычно хранят константы и общие настройки
(`currency = "RUB"`), иногда — счётчик созданных объектов.

**Как Python ищет атрибут.** `obj.attr`: сначала в словаре экземпляра (`obj.__dict__`), потом в классе,
потом в родительских классах. Поэтому присваивание `obj.attr = ...` всегда создаёт атрибут
*экземпляра* — даже если такой есть в классе: класс не меняется, а «заслоняется» для этого объекта.

Ловушка: изменяемый атрибут класса (`items = []` в теле класса) один на всех — `self.items.append(...)`
изменит общий список. Списки и словари для каждого объекта создавайте в `__init__`.

In [40]:
class Counter:
    created = 0                     # атрибут класса: общий счётчик

    def __init__(self, start=0):
        self.value = start          # атрибут экземпляра
        Counter.created += 1        # меняем именно атрибут класса

    def increment(self):
        self.value += 1
        return self.value

a, b = Counter(), Counter(10)
a.increment(); a.increment(); b.increment()
print(a.value, b.value, Counter.created, a.created)
print(a.__dict__)                   # только атрибуты экземпляра
print("created" in Counter.__dict__, "created" in a.__dict__)

class Bad:
    items = []                      # общий список на все экземпляры!

    def add(self, x):
        self.items.append(x)

x, y = Bad(), Bad()
x.add(1)
print(y.items)                      # [1] — сюрприз

2 11 2 2
{'value': 2}
True False
[1]


#### Мини-задача 10. класс Point

**Условие.** Напишите класс `Point` с конструктором `__init__(self, x, y)` и двумя методами: `distance_to(self, other)` возвращает расстояние до другой точки по формуле √((x₁−x₂)² + (y₁−y₂)²); `move(self, dx, dy)` сдвигает точку на `dx` и `dy`, изменяя её координаты (ничего не возвращает).

**Что проверяется.** `__init__` и `self`; метод, принимающий другой объект; метод, меняющий состояние.

**Пример.**

```python
p = Point(0, 0)
print(p.distance_to(Point(3, 4)))   # 5.0
p.move(2, -3)
print(p.x, p.y)                    # 2 -3
```

In [ ]:
from tests_functions_oop import test_point

class Point:
    def __init__(self, x, y):
        ...

    def distance_to(self, other):
        ...

    def move(self, dx, dy):
        ...

test_point(Point)

### Взаимодействие объектов и композиция

Объекты редко живут поодиночке. Метод может принимать другой объект (`p.distance_to(q)`), а поле
может *содержать* другой объект: плейлист хранит список песен, банк — словарь счетов, заказ —
список товаров. Это **композиция**: объект «состоит из» других объектов (отношение «has-a»).
Объект-контейнер управляет своими частями — добавляет, ищет, считает итоги, — а части остаются
самостоятельными классами со своим поведением.

Хороший признак дизайна: каждый класс отвечает за одно. `Song` знает свою длительность, `Playlist`
знает, как складывать длительности песен. Не сваливайте всё в один класс.

In [42]:
class Song:
    def __init__(self, title, duration):
        self.title = title
        self.duration = duration      # секунды

class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []               # композиция: плейлист владеет списком песен

    def add(self, song):
        self.songs.append(song)

    def total_duration(self):
        return sum(song.duration for song in self.songs)

morning = Playlist("Утро")
morning.add(Song("Yesterday", 125))
morning.add(Song("Help!", 139))
print(morning.name, len(morning.songs), morning.total_duration())
print(morning.songs[0].title)         # объект внутри объекта

Утро 2 264
Yesterday


**Проверьте себя.** Что лучше хранить в `Playlist`: список названий песен (строк) или список
объектов `Song`?

<details><summary>Ответ</summary>
Объекты: у них есть длительность и другие данные, и класс <code>Song</code> может расти
(добавить исполнителя, альбом) без изменений в <code>Playlist</code>. Строка «знает» только своё
название.
</details>

#### Мини-задача 11. классы Song и Playlist

**Условие.** Напишите класс `Song(title, duration)` с двумя атрибутами и класс `Playlist(name)` с атрибутами `name` и `songs` (пустой список) и методами: `add(self, song)` добавляет песню; `total_duration(self)` возвращает сумму длительностей; `longest(self)` возвращает объект самой длинной песни или `None`, если песен нет.

**Что проверяется.** композиция (объекты внутри объекта); `max` с `key`; обработка пустого случая.

**Пример.**

```python
p = Playlist("Утро")
p.add(Song("A", 100))
p.add(Song("B", 250))
print(p.total_duration())        # 350
print(p.longest().title)        # B
print(Playlist("X").longest())   # None
```

In [ ]:
from tests_functions_oop import test_playlist

class Song:
    def __init__(self, title, duration):
        ...


class Playlist:
    def __init__(self, name):
        ...

    def add(self, song):
        ...

    def total_duration(self):
        ...

    def longest(self):
        ...

test_playlist(Playlist, Song)

## 5.2. Магические методы, переопределение методов. Наследование

### Магические методы: как встроить объект в язык

Методы вида `__имя__` Python вызывает сам в ответ на операторы и встроенные функции:
`print(obj)` → `obj.__str__()`, `a == b` → `a.__eq__(b)`, `len(obj)` → `obj.__len__()`,
`obj[i]` → `obj.__getitem__(i)`. Определив их в классе, вы «встраиваете» свой объект в язык:
его можно печатать, сравнивать, сортировать, перебирать в `for`. `__init__` — тоже магический метод.

| Что пишем | Что вызывается | Зачем |
|---|---|---|
| `str(obj)`, `print(obj)`, f-строка | `__str__` | понятный текст для человека |
| `repr(obj)`, объект внутри списка | `__repr__` | однозначное представление для отладки, похожее на код создания |
| `a == b`, `a != b` | `__eq__` | сравнение по содержимому (по умолчанию — по идентичности) |
| `a < b`, `sorted`, `min`, `max` | `__lt__` | порядок |
| `len(obj)` | `__len__` | размер, целое ≥ 0 |
| `obj[key]` | `__getitem__` | доступ по индексу или ключу |
| `x in obj` | `__contains__` | проверка вхождения |
| `for x in obj` | `__iter__` | перебор |
| `a + b`, `abs(a)` | `__add__`, `__abs__` | арифметика |
| `bool(obj)`, `if obj:` | `__bool__` (или `__len__`) | истинность |

**Как это выполняет Python.** `a == b`: сначала пробует `type(a).__eq__(a, b)`; если тот вернул
`NotImplemented`, пробует `b.__eq__(a)`; если и это не помогло — сравнивает идентичность (`is`).
`!=` автоматически выводится из `__eq__`. `print(obj)` без `__str__` использует `__repr__`, а без
обоих печатает `<__main__.Point object at 0x...>`.

In [44]:
class Money:
    def __init__(self, amount, currency="RUB"):
        self.amount = amount
        self.currency = currency

    def __str__(self):
        return f"{self.amount} {self.currency}"

    def __repr__(self):
        return f"Money({self.amount!r}, {self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented          # «не знаю, как сравнивать» — пусть Python решит сам
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __lt__(self, other):
        return self.amount < other.amount

    def __add__(self, other):
        return Money(self.amount + other.amount, self.currency)

wallet = [Money(300), Money(50), Money(120)]
print(wallet[0])                  # __str__
print(wallet)                     # список показывает элементы через __repr__
print(Money(50) == Money(50), Money(50) == 50, Money(50) != Money(70))
print(sorted(wallet))             # __lt__
print(sum(wallet, Money(0)))      # __add__

300 RUB
[Money(300, 'RUB'), Money(50, 'RUB'), Money(120, 'RUB')]
True False True
[Money(50, 'RUB'), Money(120, 'RUB'), Money(300, 'RUB')]
470 RUB


**Типичные ошибки**

- `__str__` возвращает не строку → `TypeError`. `__str__` должен вернуть строку, а не напечатать её.
- `__eq__` без проверки типа: `Money(1) == 1` → `AttributeError`. Возвращайте `NotImplemented`
  для чужих типов.
- Определили `__eq__`, но не `__lt__` — `sorted` бросит `TypeError: '<' not supported`.
- Определив `__eq__`, вы теряете хеш: объекты больше нельзя класть в `set` и использовать как ключи
  `dict` (Python делает `__hash__ = None`). Если это нужно — определите и `__hash__`.

**Проверьте себя.** Что напечатает `print([Money(1)])`, если в классе есть только `__str__`?

<details><summary>Ответ</summary>
<code>[&lt;__main__.Money object at 0x...&gt;]</code>: список показывает элементы через
<code>__repr__</code>, а его нет. Поэтому <code>__repr__</code> стоит определять всегда.
</details>

#### Мини-задача 12. класс Vector

**Условие.** Напишите класс `Vector(x, y)` с магическими методами: `__add__` возвращает новый вектор — сумму координат; `__eq__` сравнивает координаты (для других типов — `NotImplemented`); `__str__` → `"(3, -4)"`; `__abs__` → длина вектора √(x² + y²).

**Что проверяется.** арифметический магический метод, возвращающий новый объект; `__eq__` с `NotImplemented`; `__str__`; `__abs__`.

**Пример.**

```python
v = Vector(1, 2) + Vector(3, 4)
print(v)                              # (4, 6)
print(v == Vector(4, 6))              # True
print(abs(Vector(3, 4)))              # 5.0
```

In [ ]:
from tests_functions_oop import test_vector

class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        ...

    def __eq__(self, other):
        ...

    def __str__(self):
        ...

    def __abs__(self):
        ...

test_vector(Vector)

### Наследование и переопределение методов

Наследование позволяет создать класс **на основе** другого: `class Dog(Animal)`. `Dog` получает
все методы и атрибуты `Animal` (родителя, базового класса) и может **добавить** свои или
**переопределить** унаследованные — определить метод с тем же именем заново. Это отношение «is-a»:
собака — это животное; `isinstance(dog, Animal)` → `True`.

`super()` даёт доступ к методам родителя из подкласса. Чаще всего — в `__init__`:
`super().__init__(name)` выполняет инициализацию родителя, а потом подкласс добавляет свои поля.
Без этого вызова поля родителя не появятся.

**Как Python ищет метод.** `dog.speak()`: сначала в классе `Dog`, потом в `Animal`, потом в `object`.
Цепочка поиска — `Dog.__mro__`. Переопределённый метод «заслоняет» родительский, но тот остаётся
доступен через `super().speak()`.

**Полиморфизм** — один и тот же вызов `animal.speak()` даёт разное поведение в зависимости от класса
объекта. Код, который работает со списком животных, не проверяет тип каждого: он просто вызывает
метод, а объект сам знает, что делать. В Python полиморфизм даже не требует наследования —
достаточно, чтобы у объектов был метод с нужным именем («утиная типизация»).

In [46]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def introduce(self):
        return f"{self.name} говорит: {self.speak()}"      # speak берётся у класса самого объекта

class Dog(Animal):
    def speak(self):                          # переопределение
        return "гав"

class Cat(Animal):
    def __init__(self, name, indoor=True):
        super().__init__(name)                # инициализация родителя
        self.indoor = indoor

    def speak(self):
        return "мяу"

    def introduce(self):
        return super().introduce() + (" (домашний)" if self.indoor else "")   # расширяем, а не заменяем

animals = [Dog("Рекс"), Cat("Мурка"), Animal("Кто-то")]
for animal in animals:
    print(animal.introduce())                 # полиморфизм: один вызов — разное поведение

print(isinstance(animals[0], Animal), isinstance(animals[0], Cat))
print([cls.__name__ for cls in Cat.__mro__])

Рекс говорит: гав
Мурка говорит: мяу (домашний)
Кто-то говорит: ...
True False
['Cat', 'Animal', 'object']


**Композиция или наследование?** Наследование — когда подкласс *является* частным случаем родителя
и хочет всё его поведение: `SavingsAccount` — это `BankAccount`. Композиция — когда объект
*содержит* другой или *использует* его: `Bank` содержит счета, но банк — не счёт. Если сомневаетесь,
начинайте с композиции: её проще менять. Глубокие иерархии наследования — частый источник
запутанного кода.

**Типичные ошибки**

- Забыли `super().__init__(...)` → `AttributeError` на полях родителя.
- Переопределили метод с другим набором параметров — код, рассчитанный на родителя, ломается.
- Наследование ради того, чтобы «взять пару методов», там, где нужна композиция.

**Проверьте себя.** Что напечатает код?

```python
class A:
    def f(self):
        return "A"
    def g(self):
        return self.f()

class B(A):
    def f(self):
        return "B"

print(B().g())
```

<details><summary>Ответ</summary>
<code>B</code>: метод <code>g</code> унаследован от <code>A</code>, но <code>self.f()</code> ищется
начиная с класса самого объекта — <code>B</code>.
</details>

#### Мини-задача 13. Shape, Rectangle, Circle

**Условие.** Напишите базовый класс `Shape(name)`: атрибут `name`; метод `area(self)` бросает `NotImplementedError` (у абстрактной фигуры площади нет); метод `describe(self)` возвращает `f"{name}: площадь {area:.2f}"`, вызывая `self.area()`. Затем два подкласса: `Rectangle(width, height)` с именем `"прямоугольник"` и `Circle(radius)` с именем `"круг"` (площадь `math.pi * r ** 2`). В подклассах вызывайте конструктор родителя через `super().__init__(...)` и переопределяйте только `area` — `describe` должен работать полиморфно.

**Что проверяется.** наследование; `super().__init__`; переопределение метода; полиморфный вызов из метода родителя; `NotImplementedError`.

**Ограничения.** в `__init__` подклассов должен быть вызов `super()` (проверяется).

**Пример.**

```python
shapes = [Rectangle(3, 4), Circle(2)]
for s in shapes:
    print(s.describe())
# прямоугольник: площадь 12.00
# круг: площадь 12.57
```

In [ ]:
from tests_functions_oop import test_shapes

import math


class Shape:
    def __init__(self, name):
        ...

    def area(self):
        ...

    def describe(self):
        ...


class Rectangle(Shape):
    def __init__(self, width, height):
        ...

    def area(self):
        ...


class Circle(Shape):
    def __init__(self, radius):
        ...

    def area(self):
        ...

test_shapes(Shape, Rectangle, Circle)

## 5.3. Модель исключений Python. `try`, `except`, `else`, `finally`. Модули

### Исключения: что это и зачем

Исключение — это сигнал об ошибке, который прерывает обычное выполнение: `int("abc")` →
`ValueError`, `[1][5]` → `IndexError`, `{}["k"]` → `KeyError`, `1 / 0` → `ZeroDivisionError`.
Если исключение никто не обработал, программа завершается с трассировкой.

**Как Python это выполняет.** Возникшее исключение «поднимается» по стеку вызовов: из функции, где
случилось, в вызвавшую её, и так далее, пока не встретит подходящий `except`. Всё, что после точки
ошибки в этих функциях, не выполняется. Так ошибка из глубины программы может быть обработана там,
где известно, что с ней делать.

```python
try:
    рискованный_код()
except ValueError as e:       # если внутри try возникло ValueError (или его подкласс); e — объект исключения
    обработать(e)
except (TypeError, KeyError): # несколько типов сразу
    ...
else:                         # выполняется, если исключения НЕ было
    ...
finally:                      # выполняется всегда: и после успеха, и после ошибки — закрыть файл, соединение
    ...
```

`raise ValueError("сообщение")` — создать исключение самому: так функция сообщает, что её вызвали
с плохими данными. Лучше упасть с понятным сообщением, чем молча вернуть неверный результат.

In [48]:
def parse_age(text):
    try:
        age = int(text)
    except ValueError:
        return None            # не число
    else:
        if age < 0:
            raise ValueError(f"возраст не может быть отрицательным: {age}")
        return age
    finally:
        print(f"  (проверили {text!r})")

print(parse_age("20"))
print(parse_age("abc"))
try:
    parse_age("-5")
except ValueError as e:
    print("ValueError:", e)

  (проверили '20')
20
  (проверили 'abc')
None
  (проверили '-5')
ValueError: возраст не может быть отрицательным: -5


### Стандартные исключения и свои классы

Стандартные исключения образуют иерархию: `Exception` → `ValueError`, `TypeError`,
`LookupError` → (`KeyError`, `IndexError`), `ArithmeticError` → `ZeroDivisionError`,
`OSError` → `FileNotFoundError`… `except LookupError` ловит и `KeyError`, и `IndexError`.
`except Exception` ловит почти всё — и это редко правильно: вы скроете и опечатки в собственном коде.

Своё исключение — это просто класс, унаследованный от `Exception` (или от более подходящего
стандартного: `class NegativeNumberError(ValueError)`). Тело может быть пустым — `pass` или docstring.
Зачем: вызывающий код сможет отличать *ваши* ошибки от чужих и обрабатывать их по-разному:
`except InsufficientFundsError:` — показать сообщение пользователю, а остальное не трогать.

Хорошая практика — базовый класс для всех ошибок вашего модуля (`class LibraryError(Exception)`)
и конкретные подклассы. Тогда «любую ошибку библиотеки» можно поймать одним `except LibraryError`.

In [49]:
class BankError(Exception):
    """Базовая ошибка банка."""

class InsufficientFundsError(BankError):
    def __init__(self, balance, amount):
        super().__init__(f"недостаточно средств: есть {balance}, нужно {amount}")
        self.balance = balance
        self.amount = amount

def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError("сумма должна быть положительной")
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

for amount in (30, 500, -1):
    try:
        print("остаток:", withdraw(100, amount))
    except InsufficientFundsError as e:
        print("своя ошибка:", e, "| не хватает:", e.amount - e.balance)
    except ValueError as e:
        print("ValueError:", e)

print(issubclass(InsufficientFundsError, BankError), issubclass(KeyError, LookupError))

остаток: 70
своя ошибка: недостаточно средств: есть 100, нужно 500 | не хватает: 400
ValueError: сумма должна быть положительной
True True


**Типичные ошибки**

- Голый `except:` или `except Exception:` вокруг большого куска кода — прячет настоящие ошибки.
  Ловите конкретные типы и оборачивайте минимальный участок кода.
- `except ValueError, TypeError:` — старый синтаксис; нужны скобки: `except (ValueError, TypeError):`.
- Забыть `raise`: строка `ValueError("...")` без `raise` создаёт объект исключения и выбрасывает его
  как обычное значение.
- Проверять вместо обработки там, где проще попробовать: `try: v = d[key] except KeyError:` —
  стиль EAFP («проще попросить прощения, чем разрешения»). Оба подхода допустимы, но `try` часто
  короче и надёжнее.

**Проверьте себя.** Что напечатает код?

```python
def f():
    try:
        return "try"
    finally:
        print("finally")

print(f())
```

<details><summary>Ответ</summary>
Сначала <code>finally</code>, потом <code>try</code>: блок <code>finally</code> выполняется даже
тогда, когда из <code>try</code> выходят через <code>return</code>.
</details>

#### Мини-задача 14. safe_sqrt и NegativeNumberError

**Условие.** Объявите класс исключения `NegativeNumberError`, унаследованный от `ValueError`. Напишите функцию `safe_sqrt(x)`, которая возвращает квадратный корень из `x` (`x ** 0.5`), а для отрицательного `x` бросает `NegativeNumberError` с сообщением, в котором есть само число.

**Что проверяется.** свой класс исключения; `raise` с сообщением; выбор родителя для исключения.

**Пример.**

```python
print(safe_sqrt(16))        # 4.0
try:
    safe_sqrt(-9)
except ValueError as e:      # NegativeNumberError — это ValueError
    print("ошибка:", e)     # ошибка: ... -9 ...
```

In [ ]:
from tests_functions_oop import test_safe_sqrt

class NegativeNumberError(ValueError):
    ...


def safe_sqrt(x):
    ...

test_safe_sqrt(safe_sqrt, NegativeNumberError)

#### Мини-задача 15. read_numbers

**Условие.** Напишите функцию `read_numbers(strings)`, которая получает список строк и возвращает список целых чисел, полученных из тех строк, которые удалось преобразовать через `int()`. Строки, которые не преобразуются (`"abc"`, `"3.5"`, `""`), просто пропускаются. Обработайте ошибку через `try`/`except`: заранее проверить строку методами вроде `isdigit()` не получится — `"-5"` и `" 4 "` он не пропустит, а `int()` их понимает.

**Что проверяется.** `try`/`except` вокруг минимального участка кода; конкретный тип исключения; стиль EAFP.

**Ограничения.** в решении должен быть `try`/`except` (проверяется).

**Пример.**

```python
>>> read_numbers(['10', 'abc', '-5', '3.5', '7'])
[10, -5, 7]
>>> read_numbers([])
[]
```

In [ ]:
from tests_functions_oop import test_read_numbers

def read_numbers(strings):
    ...

test_read_numbers(read_numbers)

### Модули: `import`, `from ... import`, свои модули

Модуль — это файл `.py`. `import math` выполняет файл модуля один раз, создаёт объект-модуль
и привязывает его к имени; дальше — `math.sqrt(2)`. `from math import sqrt, pi` копирует отдельные
имена в ваше пространство имён; `import numpy as np` — псевдоним. `from math import *` не используйте:
непонятно, откуда взялись имена.

**Свой модуль** — любой ваш файл `.py` в той же папке (или в папке из `sys.path`). Если рядом
с ноутбуком лежит `helpers.py` с функцией `slugify`, то `from helpers import slugify` работает.
Так проект делится на части: `models.py` (классы), `services.py` (логика), `main.py` (запуск).
Папка с файлом `__init__.py` — **пакет**: `from shop.models import Product`.

**Как Python выполняет `import`.** При первом импорте файл выполняется целиком сверху вниз: все `def`
и `class` создают объекты, а код верхнего уровня (например, `print`) тоже выполняется. Результат
кэшируется в `sys.modules`: повторный `import` ничего не выполняет заново. В ноутбуке после правки
файла нужен `importlib.reload(module)` или перезапуск ядра.

`if __name__ == "__main__":` — способ отличить «файл запустили как программу» (`python helpers.py`,
тогда `__name__ == "__main__"`) от «файл импортировали» (тогда `__name__ == "helpers"`). Под этим
условием пишут демонстрацию или запуск программы, чтобы при импорте они не срабатывали.

Команда `%%writefile имя.py` в первой строке ячейки сохраняет ячейку в файл рядом с ноутбуком —
так можно создавать модули, не выходя из Jupyter.

In [52]:
%%writefile helpers.py
"""Маленький модуль-пример."""

GREETING = "Привет"

def slugify(text):
    """'Hello World' -> 'hello-world'"""
    return "-".join(text.lower().split())

if __name__ == "__main__":
    print("запущен как программа:", slugify("Hello World"))

Writing helpers.py


In [53]:
import importlib
import helpers                          # выполняет helpers.py один раз; print под __main__ не сработал
importlib.reload(helpers)               # на случай, если файл менялся после первого импорта

from helpers import slugify, GREETING
print(helpers.__name__, slugify("Функции и ООП"), GREETING)
print(helpers.slugify is slugify)

import math as m                        # псевдоним
print(m.sqrt(16), [name for name in dir(helpers) if not name.startswith("_")])

helpers функции-и-ооп Привет
True
4.0 ['GREETING', 'slugify']


In [54]:
!python helpers.py

запущен как программа: hello-world


**Типичные ошибки**

- `ModuleNotFoundError`: файл не в той папке, опечатка в имени, или ваш файл называется как стандартный
  модуль (`random.py` рядом с ноутбуком «перекроет» настоящий `random`).
- Код верхнего уровня в модуле (печать, запуск) срабатывает при импорте — прячьте его под
  `if __name__ == "__main__":`.
- Циклический импорт: `a.py` импортирует `b.py`, а тот — `a.py`. Выносите общее в третий модуль.

**Проверьте себя.** В файле `tools.py` есть строка `print("загружен")`. Сколько раз она выполнится,
если написать `import tools` дважды подряд?

<details><summary>Ответ</summary>
Один раз: после первого импорта модуль лежит в <code>sys.modules</code>, и второй
<code>import</code> берёт его оттуда, не выполняя файл заново.
</details>

#### Мини-задача 16. модуль shapes.py

**Условие.** Создайте модуль `shapes.py` (ячейка ниже начинается с `%%writefile shapes.py` — при запуске она сохранит файл рядом с ноутбуком). В модуле: функции `rectangle_area(width, height)` и `circle_area(radius)` и блок `if __name__ == "__main__":` с демонстрацией (например, напечатать площади). При импорте модуль не должен ничего печатать. Затем запустите ячейку с тестом: он импортирует модуль и проверит функции и защиту блока `__main__`.

**Что проверяется.** создание своего модуля; `if __name__ == "__main__"`; импорт своего кода.

**Пример.**

```python
import shapes
print(shapes.rectangle_area(3, 4))   # 12
```

In [ ]:
%%writefile shapes.py
import math


def rectangle_area(width, height):
    ...


def circle_area(radius):
    ...


if __name__ == "__main__":
    ...

In [ ]:
from tests_functions_oop import test_shapes_module

test_shapes_module()

### Итоги главы 5 (теория)

| Тема | Главное |
|---|---|
| Класс и экземпляр | класс — чертёж, экземпляр — объект; `Point(1, 2)` создаёт объект и вызывает `__init__` |
| `self` | объект, для которого вызван метод; Python подставляет его сам |
| Атрибуты | экземпляра — через `self.x = ...`; класса — в теле класса, общие для всех |
| Композиция | объект хранит другие объекты («has-a»); каждый класс отвечает за одно |
| Магические методы | `__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__getitem__`, `__iter__` встраивают объект в язык |
| Наследование | `class B(A)`: «is-a»; переопределение; `super()`; полиморфизм |
| Исключения | `try/except/else/finally`; `raise`; свои классы от `Exception`; ловите конкретные типы |
| Модули | файл `.py`; `import`; `if __name__ == "__main__"`; проект из нескольких файлов |

Дальше — практика: три уровня вокруг банковских счетов и итоговый проект.

## 5.4. Уровень 1 — собираем класс `BankAccount`

На этом уровне вы построите один настоящий класс — банковский счёт — за семь шагов. Каждый шаг
добавляет что-то к классу из предыдущего: **скопируйте класс из предыдущего шага в ячейку нового
шага и допишите новое**. К седьмому шагу получится полноценный класс с проверками, историей операций,
сравнением и переводами — он понадобится на уровнях 2 и 3.

Тесты каждого шага проверяют новое поведение и немного старого. Если тест шага 3 падает на чём-то
из шага 2 — при копировании что-то потерялось.

Договорённости на весь уровень:

- суммы — `int` или `float`; в примерах в основном целые;
- имена методов и атрибутов — ровно как в условии: тесты обращаются к ним по именам;
- в примерах к шагам код показан с комментариями-результатами, а не как `>>>`-сессия.

### Шаг 1. Класс и `__init__`

**Условие.** Создайте класс `BankAccount` с конструктором `__init__(self, owner)`, который сохраняет имя владельца в атрибут `owner`.

**Что проверяется.** объявление класса; `__init__` и `self`; атрибут экземпляра.

**Пример.**

```python
acc = BankAccount("Аня")
print(acc.owner)          # Аня
```

In [ ]:
from tests_functions_oop import test_account_step1

class BankAccount:
    def __init__(self, owner):
        ...

test_account_step1(BankAccount)

### Шаг 2. Основные поля

**Условие.** Добавьте поля: параметр `balance=0` в конструктор и одноимённый атрибут; атрибут `history` — пустой список для будущих операций (создаётся в `__init__`, свой у каждого счёта); атрибут **класса** `currency = "RUB"`, общий для всех счетов.

**Что проверяется.** параметр по умолчанию в `__init__`; атрибуты экземпляра и атрибут класса; ловушка общего изменяемого объекта — списки создаём в `__init__`, а не в теле класса.

**Пример.**

```python
acc = BankAccount("Аня", 250)
print(acc.balance, acc.history)          # 250 []
print(acc.currency, BankAccount.currency)  # RUB RUB
print(BankAccount("Боря").balance)      # 0
```

In [ ]:
from tests_functions_oop import test_account_step2

# Скопируйте сюда класс BankAccount из шага 1 и дополните его.

class BankAccount:
    ...

test_account_step2(BankAccount)

### Шаг 3. Методы `deposit` и `withdraw`

**Условие.** Добавьте методы `deposit(self, amount)` и `withdraw(self, amount)`: первый увеличивает баланс на `amount`, второй уменьшает. Оба **возвращают новый баланс**. Проверок пока нет — они появятся на шаге 5.

**Что проверяется.** методы, меняющие состояние через `self`; возврат значения из метода.

**Пример.**

```python
acc = BankAccount("Аня", 50)
print(acc.deposit(100))    # 150
print(acc.withdraw(30))    # 120
print(acc.balance)         # 120
```

In [ ]:
from tests_functions_oop import test_account_step3

# Скопируйте сюда класс BankAccount из шага 2 и добавьте методы.

class BankAccount:
    ...

test_account_step3(BankAccount)

### Шаг 4. История операций

**Условие.** Сделайте так, чтобы каждая операция записывалась в `history`: `deposit` добавляет кортеж `("deposit", amount)`, `withdraw` — `("withdraw", amount)`. Начальный баланс операцией не считается. Добавьте методы `total_deposited(self)` и `total_withdrawn(self)`, которые суммируют суммы операций соответствующего вида по истории.

**Что проверяется.** изменение состояния объекта; список кортежей; обход истории с распаковкой; генераторное выражение внутри `sum`.

**Пример.**

```python
acc = BankAccount("Аня", 100)
acc.deposit(100)
acc.withdraw(30)
print(acc.history)                                  # [('deposit', 100), ('withdraw', 30)]
print(acc.total_deposited(), acc.total_withdrawn())  # 100 30
```

In [ ]:
from tests_functions_oop import test_account_step4

# Скопируйте сюда класс BankAccount из шага 3 и дополните его.

class BankAccount:
    ...

test_account_step4(BankAccount)

### Шаг 5. Проверка входных данных и своё исключение

**Условие.** Добавьте проверки входных данных.

- Объявите класс исключения `InsufficientFundsError(Exception)`.
- `deposit` и `withdraw`: если `amount` не `int` и не `float` — `TypeError`; если `amount <= 0` — `ValueError`.
- `withdraw`: если `amount` больше доступной суммы — `InsufficientFundsError`. Доступную сумму возвращает новый метод `available(self)` — пока это просто `self.balance` (на уровне 3 у кредитного счёта он изменится, а `withdraw` останется прежним).
- `__init__`: пустое имя владельца (или не строка) — `ValueError`; отрицательный начальный баланс — `ValueError`.

При ошибке состояние счёта не должно меняться: проверяйте *до* изменения баланса и истории. Проверку суммы вынесите в отдельный вспомогательный метод (например, `_check_amount`), чтобы не дублировать код в `deposit` и `withdraw`.

**Что проверяется.** `raise` с подходящим типом; свой класс исключения; `isinstance`; порядок «сначала проверить, потом изменить»; вспомогательный метод.

**Пример.**

```python
acc = BankAccount("Аня", 100)
try:
    acc.withdraw(500)
except InsufficientFundsError as e:
    print("нельзя:", e)
print(acc.balance, acc.history)   # 100 [] — ничего не изменилось
print(acc.available())            # 100
```

In [ ]:
from tests_functions_oop import test_account_step5

class InsufficientFundsError(Exception):
    """Недостаточно средств на счёте."""


# Скопируйте сюда класс BankAccount из шага 4 и добавьте проверки.

class BankAccount:
    ...

test_account_step5(BankAccount, InsufficientFundsError)

### Шаг 6. Магические методы

**Условие.** Добавьте магические методы:

- `__str__` → `"Аня: 150 RUB"` (владелец, баланс, валюта через пробелы и двоеточие);
- `__repr__` → `"BankAccount('Аня', 150)"`: используйте `type(self).__name__` вместо жёстко прописанного имени класса и `!r` для владельца — тогда подклассы на уровне 3 получат правильный `repr` бесплатно;
- `__eq__`: счета равны, если совпадают владелец и баланс; для объектов других типов возвращайте `NotImplemented`;
- `__lt__`: сравнение по балансу, чтобы работали `sorted`, `min` и `max`.

**Что проверяется.** `__str__` и `__repr__`; `__eq__` с `NotImplemented`; `__lt__` и сортировка объектов.

**Пример.**

```python
a, b = BankAccount("Аня", 300), BankAccount("Боря", 100)
print(a)                       # Аня: 300 RUB
print([b])                     # [BankAccount('Боря', 100)]
print(a == BankAccount("Аня", 300), a == "Аня")   # True False
print([acc.owner for acc in sorted([a, b])])       # ['Боря', 'Аня']
```

In [ ]:
from tests_functions_oop import test_account_step6

class InsufficientFundsError(Exception):
    """Недостаточно средств на счёте."""


# Скопируйте сюда класс BankAccount из шага 5 и добавьте магические методы.

class BankAccount:
    ...

test_account_step6(BankAccount)

### Шаг 7. Переводы и `__len__`: класс готов

**Условие.** Доведите класс до рабочего состояния:

- `transfer(self, other, amount)` — перевести `amount` на другой счёт: снять с себя (`withdraw`) и положить другому (`deposit`). Если снять нельзя, исключение из `withdraw` прервёт перевод, и ничего не изменится — поэтому снимайте *первым*;
- `__len__` — число операций в истории.

Переиспользуйте уже написанные методы: `transfer` не должен трогать `balance` напрямую — тогда проверки и запись в историю сработают сами.

**Что проверяется.** взаимодействие двух объектов одного класса; переиспользование методов; `__len__`.

**Пример.**

```python
a, b = BankAccount("Аня", 100), BankAccount("Боря", 100)
a.transfer(b, 50)
print(a.balance, b.balance)    # 50 150
print(a.history, b.history)    # [('withdraw', 50)] [('deposit', 50)]
print(len(a), len(BankAccount("Вера")))   # 1 0
```

In [ ]:
from tests_functions_oop import test_account_step7

class InsufficientFundsError(Exception):
    """Недостаточно средств на счёте."""


# Скопируйте сюда класс BankAccount из шага 6 и завершите его.

class BankAccount:
    ...

test_account_step7(BankAccount)

## 5.5. Уровень 2 — класс `Bank`

Второй класс — `Bank` — управляет многими счетами. Он **содержит** объекты `BankAccount`
(композиция), выдаёт им номера, ищет по номеру, считает итоги, переводит деньги между счетами
и ведёт себя как контейнер: `len(bank)`, `bank[1]`, `for acc in bank`.

Понадобится готовый `BankAccount` из уровня 1 (все семь шагов): ячейка с ним должна быть выполнена
в ноутбуке, иначе `open_account` не найдёт класс. `Bank` тоже собирается по шагам — копируйте его
из предыдущего шага.

### Шаг 1. Класс, словарь счетов и `open_account`

**Условие.** Создайте класс `Bank(name)` с атрибутами `name` и `accounts` — словарь `{номер: BankAccount}`, изначально пустой. Метод `open_account(self, owner, balance=0)` создаёт `BankAccount`, кладёт его в словарь под следующим свободным номером (1, 2, 3, …) и **возвращает номер**. Номера выдаются по порядку и никогда не повторяются — храните счётчик следующего номера в атрибуте объекта.

**Что проверяется.** композиция: объект создаёт и хранит другие объекты; словарь как хранилище; счётчик в атрибуте.

**Пример.**

```python
bank = Bank("PyBank")
print(bank.open_account("Аня", 300))   # 1
print(bank.open_account("Боря"))       # 2
print(bank.accounts[1].owner, bank.accounts[2].balance)   # Аня 0
```

In [ ]:
from tests_functions_oop import test_bank_step1

class Bank:
    def __init__(self, name):
        ...

    def open_account(self, owner, balance=0):
        ...

test_bank_step1(Bank)

### Шаг 2. Поиск, закрытие счёта и своё исключение

**Условие.** Объявите класс исключения `AccountNotFoundError(Exception)`. Добавьте методы:

- `get_account(self, number)` — возвращает счёт по номеру или бросает `AccountNotFoundError`;
- `close_account(self, number)` — удаляет счёт из словаря. Если счёта нет — `AccountNotFoundError`; если на нём есть деньги (`balance != 0`) — `ValueError`, и счёт остаётся. Освободившиеся номера не переиспользуются.

**Что проверяется.** поиск в словаре; своё исключение; `del`; порядок проверок.

**Пример.**

```python
bank = Bank("PyBank")
bank.open_account("Аня"); bank.open_account("Боря", 10)
bank.close_account(1)
print(1 in bank.accounts)          # False
try:
    bank.close_account(2)
except ValueError as e:
    print("нельзя:", e)
print(bank.open_account("Вера"))    # 3 — номер 1 не переиспользуется
```

In [ ]:
from tests_functions_oop import test_bank_step2

class AccountNotFoundError(Exception):
    """Счёт с таким номером не найден."""


# Скопируйте сюда класс Bank из шага 1 и дополните его.

class Bank:
    ...

test_bank_step2(Bank, AccountNotFoundError)

### Шаг 3. Магические методы контейнера

**Условие.** Сделайте банк контейнером:

- `__len__` — число счетов;
- `__contains__(self, number)` — есть ли счёт с таким номером (`number in bank`);
- `__getitem__(self, number)` — то же, что `get_account` (`bank[number]`);
- `__iter__` — перебор счетов по возрастанию номера: используйте `yield` в цикле по `sorted(self.accounts)` или верните `iter(...)` от списка;
- `__str__` → `"Банк PyBank: счетов — 3"`.

**Что проверяется.** магические методы контейнера; генератор в `__iter__`; согласованность `len`, `in`, `for`.

**Пример.**

```python
bank = Bank("PyBank")
for owner in ["Аня", "Боря", "Вера"]:
    bank.open_account(owner)
print(len(bank), 2 in bank, 9 in bank)         # 3 True False
print(bank[1].owner)                          # Аня
print([acc.owner for acc in bank])            # ['Аня', 'Боря', 'Вера']
print(bank)                                   # Банк PyBank: счетов — 3
```

In [ ]:
from tests_functions_oop import test_bank_step3

class AccountNotFoundError(Exception):
    """Счёт с таким номером не найден."""


# Скопируйте сюда класс Bank из шага 2 и добавьте магические методы.

class Bank:
    ...

test_bank_step3(Bank)

### Шаг 4. Свойство `total_balance`, поиск и сортировка

**Условие.** Добавьте:

- свойство `total_balance` (через `@property`) — сумма балансов всех счетов; читается как атрибут: `bank.total_balance`, без скобок;
- `richest(self)` — счёт с наибольшим балансом (`max` с `key`) или `None`, если счетов нет;
- `find(self, predicate)` — список счетов (в порядке номеров), для которых функция `predicate(account)` вернула истину. Так вызывающий код сам решает, что искать: `bank.find(lambda acc: acc.balance > 100)`;
- `sorted_by_balance(self, descending=False)` — список счетов по возрастанию баланса (или по убыванию при `descending=True`).

**Что проверяется.** `@property`; `max` с `key`; функция как параметр (предикат) — функции высших порядков внутри класса; `sorted` с `key` и `reverse`.

**Пример.**

```python
bank = Bank("PyBank")
bank.open_account("Аня", 300); bank.open_account("Боря", 100)
print(bank.total_balance)                                  # 400
print(bank.richest().owner)                                # Аня
print([a.owner for a in bank.find(lambda a: a.balance < 200)])   # ['Боря']
print([a.owner for a in bank.sorted_by_balance(descending=True)])  # ['Аня', 'Боря']
```

In [ ]:
from tests_functions_oop import test_bank_step4

class AccountNotFoundError(Exception):
    """Счёт с таким номером не найден."""


# Скопируйте сюда класс Bank из шага 3 и дополните его.

class Bank:
    ...

test_bank_step4(Bank)

### Шаг 5. Операции по номеру счёта

**Условие.** Добавьте методы, работающие с номерами счетов:

- `deposit(self, number, amount)` и `withdraw(self, number, amount)` — находят счёт через `get_account` и вызывают его метод; возвращают новый баланс;
- `transfer(self, from_number, to_number, amount)` — находит **оба** счёта (если какого-то нет — `AccountNotFoundError` до любых изменений) и вызывает `transfer` у счёта-источника.

Исключения счёта (`ValueError`, `TypeError`, `InsufficientFundsError`) не перехватывайте — пусть поднимаются к вызывающему коду. Банк не дублирует логику счёта, а **делегирует** ей.

**Что проверяется.** делегирование: банк не повторяет проверки счёта; исключения проходят сквозь несколько уровней вызовов.

**Пример.**

```python
bank = Bank("PyBank")
bank.open_account("Аня", 300); bank.open_account("Боря", 100)
bank.transfer(1, 2, 100)
print(bank[1].balance, bank[2].balance, bank.total_balance)   # 200 200 400
try:
    bank.transfer(2, 1, 5000)
except InsufficientFundsError as e:
    print("нельзя:", e)
```

In [ ]:
from tests_functions_oop import test_bank_step5

class AccountNotFoundError(Exception):
    """Счёт с таким номером не найден."""


# Скопируйте сюда класс Bank из шага 4 и завершите его.

class Bank:
    ...

test_bank_step5(Bank)

## 5.6. Уровень 3 — наследование, отчёты и маленькая программа

Здесь классы уровней 1 и 2 объединяются в маленькую программу. Появятся подклассы счетов,
полиморфный «конец месяца», генератор операций, отчёты через `sorted` и `lambda`, декоратор
и обработчик команд. Понадобятся `BankAccount` и `Bank` из предыдущих уровней — их ячейки должны
быть выполнены в ноутбуке.

### Шаг 1. Подклассы `SavingsAccount` и `CreditAccount`

**Условие.** 1. Добавьте в `BankAccount` метод `month_end(self)`, который ничего не делает и возвращает `0` — «конец месяца» для обычного счёта. Скопируйте класс в ячейку ниже с этим дополнением.

2. `SavingsAccount(BankAccount)` — накопительный счёт. `__init__(self, owner, balance=0, rate=0.05)` вызывает конструктор родителя через `super()` и сохраняет `rate`. `add_interest(self)` начисляет проценты: `interest = round(self.balance * self.rate, 2)`; если он больше нуля — пополняет счёт через `deposit` (так проценты попадут в историю) — и возвращает `interest`. `month_end` начисляет проценты и возвращает их.

3. `CreditAccount(BankAccount)` — счёт с кредитным лимитом. `__init__(self, owner, balance=0, credit_limit=1000)`. Переопределите `available(self)` → `balance + credit_limit`: `withdraw` родителя сам разрешит уходить в минус до лимита. Атрибут класса `fee_rate = 0.02`. `month_end`: при отрицательном балансе списывает комиссию `fee = round(-balance * fee_rate, 2)` — уменьшает `balance` на `fee`, добавляет в историю `("fee", fee)` и возвращает `fee`; при неотрицательном балансе возвращает `0`.

Обратите внимание: `withdraw` менять не нужно — достаточно переопределить `available`. Ради этого проверка и была вынесена в отдельный метод на шаге 5 уровня 1. А `repr` подклассов покажет правильное имя благодаря `type(self).__name__`.

**Что проверяется.** наследование и `super().__init__`; переопределение методов; переиспользование методов родителя; полиморфный `month_end`.

**Ограничения.** в `__init__` обоих подклассов должен быть вызов `super()` (проверяется).

**Пример.**

```python
s = SavingsAccount("Аня", 1000)
print(s.add_interest(), s.balance, s.history)   # 50.0 1050.0 [('deposit', 50.0)]
c = CreditAccount("Боря", 100, credit_limit=500)
print(c.withdraw(400), c.available())        # -300 300
print(c.month_end(), c.balance)                 # 6.0 -306.0
print(repr(s))                                  # SavingsAccount('Аня', 1050.0)
```

In [ ]:
from tests_functions_oop import test_level3_step1

class InsufficientFundsError(Exception):
    """Недостаточно средств на счёте."""


# Скопируйте сюда BankAccount из уровня 1 и добавьте метод month_end.

class BankAccount:
    ...


class SavingsAccount(BankAccount):
    ...


class CreditAccount(BankAccount):
    ...

test_level3_step1(BankAccount, SavingsAccount, CreditAccount)

### Шаг 2. Банк принимает любые счета и закрывает месяц

**Условие.** Дополните `Bank`:

- `add_account(self, account)` — принимает готовый объект счёта (экземпляр `BankAccount` или любого его подкласса; иначе `TypeError`), присваивает следующий номер, сохраняет и возвращает номер. Перепишите `open_account` через `add_account`, чтобы номера выдавались в одном месте;
- `month_end(self)` — вызывает `month_end()` у каждого счёта в порядке номеров и возвращает список результатов.

Банк не знает, какой именно счёт перед ним: обычный, накопительный или кредитный. Он просто вызывает `month_end`, а объект сам решает, что делать. Это и есть полиморфизм.

**Что проверяется.** полиморфизм через общий интерфейс; `isinstance` с базовым классом; переиспользование кода внутри класса.

**Пример.**

```python
bank = Bank("PyBank")
bank.open_account("Аня", 100)
bank.add_account(SavingsAccount("Боря", 1000))
print(bank.month_end())                     # [0, 50.0]
print([acc.balance for acc in bank])       # [100, 1050.0]
```

In [ ]:
from tests_functions_oop import test_level3_step2

class AccountNotFoundError(Exception):
    """Счёт с таким номером не найден."""


# Скопируйте сюда Bank из уровня 2 и дополните его.

class Bank:
    ...

test_level3_step2(Bank, SavingsAccount, CreditAccount)

### Шаг 3. Генератор операций и отчёты

**Условие.** Напишите три функции (не методы — они работают с банком снаружи):

- `iter_operations(bank)` — **генератор**: по счетам в порядке номеров, по операциям каждого счёта — отдаёт кортежи `(номер, операция, сумма)`;
- `top_accounts(bank, n=3)` — список владельцев `n` самых богатых счетов по убыванию баланса; при равных балансах раньше идёт счёт с меньшим номером. Используйте `sorted` с `lambda` и составным ключом (подсказка: `bank.accounts.items()` даёт пары `(номер, счёт)`);
- `operations_summary(bank)` — словарь `{операция: суммарная сумма}` по всем операциям банка, построенный **через `iter_operations`**.

**Что проверяется.** генератор с вложенными циклами; `sorted` с составным ключом и `lambda`; словарь-счётчик; генератор как источник данных для другой функции.

**Ограничения.** `iter_operations` — функция с `yield`; в `top_accounts` есть `lambda` (проверяется).

**Пример.**

```python
bank = Bank("PyBank")
bank.open_account("Аня", 300); bank.open_account("Боря", 100)
bank.deposit(2, 50); bank.transfer(1, 2, 20)
print(list(iter_operations(bank)))
# [(1, 'withdraw', 20), (2, 'deposit', 50), (2, 'deposit', 20)]
print(top_accounts(bank, n=1))            # ['Аня']
print(operations_summary(bank))          # {'withdraw': 20, 'deposit': 70}
```

In [ ]:
from tests_functions_oop import test_level3_step3

def iter_operations(bank):
    ...


def top_accounts(bank, n=3):
    ...


def operations_summary(bank):
    ...

test_level3_step3(iter_operations, top_accounts, operations_summary, Bank)

### Шаг 4. Декоратор `catch_errors` и обработчик команд

**Условие.** Соберите всё в маленькую программу.

- `catch_errors(func)` — декоратор: обёртка вызывает `func` и возвращает результат; если внутри возникло любое исключение — возвращает строку `"ошибка: <текст исключения>"`. Используйте `functools.wraps`.
- `run_commands(bank, commands)` — обработчик команд. `commands` — список кортежей: `("open", owner, balance)` или `("open", owner)`, `("deposit", number, amount)`, `("withdraw", number, amount)`, `("transfer", from_number, to_number, amount)`, `("balance", number)`. Для каждой команды вызывается соответствующий метод банка с остальными элементами кортежа как аргументами (распаковка `*args`); результат превращается в строку через `str` и добавляется в список ответов. Неизвестная команда → `"ошибка: неизвестная команда <имя>"`. Ошибки не должны прерывать обработку — оборачивайте вызов в `catch_errors`.

Подсказка: словарь `{имя команды: метод банка}` — методы как значения словаря; для `balance` подойдёт `lambda number: bank[number].balance`. Разбирать кортеж удобно так: `for name, *args in commands`.

**Что проверяется.** декоратор с `try`/`except`; функции и методы как значения словаря; распаковка `*args` при вызове; программа, которая не падает от одной ошибки.

**Ограничения.** `catch_errors` использует `wraps` (проверяется).

**Пример.**

```python
bank = Bank("PyBank")
print(run_commands(bank, [
    ("open", "Аня", 100),
    ("withdraw", 1, 500),
    ("deposit", 7, 10),
    ("balance", 1),
    ("fly", 1),
]))
# ['1', 'ошибка: недостаточно средств: ...', 'ошибка: счёт №7 не найден', '100', 'ошибка: неизвестная команда fly']
```

In [ ]:
from tests_functions_oop import test_level3_step4

from functools import wraps


def catch_errors(func):
    ...


def run_commands(bank, commands):
    ...

test_level3_step4(catch_errors, run_commands, Bank)

## 5.7. Итоговый проект — библиотека

Итоговый проект — учёт книг в библиотеке. Это новая предметная область, и в ней нужно применить всё
сразу: два класса-«данные» (`Book`, `Reader`), класс-контейнер (`Library`), свои исключения,
функции-отчёты с `lambda`, генератор, `*args` и `**kwargs`, обработку ошибок.

Проект собирается из пяти компонентов, каждый — в своей ячейке со своим тестом. Так устроены
настоящие программы: ошибки, модели данных, логика, отчёты, сценарий запуска. В файлах это были бы
`exceptions.py`, `models.py`, `library.py`, `reports.py`, `main.py` — в ноутбуке роль файлов играют
ячейки. Компоненты зависят друг от друга по порядку: `Library` использует исключения и `Book`,
отчёты — `Library`. Выполняйте ячейки сверху вниз.

### Компонент 1. Исключения библиотеки

**Условие.** Объявите базовый класс `LibraryError(Exception)` и три его подкласса: `BookNotFoundError`, `BookNotAvailableError`, `ReaderLimitError`. Подклассы наследуются от `LibraryError`, а не друг от друга. Тело каждого — docstring с описанием ситуации.

**Что проверяется.** иерархия своих исключений; базовый класс, чтобы ловить «любую ошибку библиотеки» одним `except`.

**Пример.**

```python
try:
    raise ReaderLimitError("лимит")
except LibraryError as e:
    print("ошибка библиотеки:", e)   # ошибка библиотеки: лимит
```

In [ ]:
from tests_functions_oop import test_project_exceptions

class LibraryError(Exception):
    """Базовая ошибка библиотеки."""


class BookNotFoundError(LibraryError):
    ...


class BookNotAvailableError(LibraryError):
    ...


class ReaderLimitError(LibraryError):
    ...

test_project_exceptions(LibraryError, BookNotFoundError, BookNotAvailableError, ReaderLimitError)

### Компонент 2. Класс `Book`

**Условие.** Напишите класс `Book(title, author, year)`:

- проверки в `__init__`: `title` и `author` — непустые строки (строка из одних пробелов не считается), иначе `ValueError`; `year` — `int` от 1450 до 2100 включительно, иначе `ValueError`;
- атрибут `available = True` — книга свободна;
- `__str__` → `"Идиот — Достоевский (1869)"`;
- `__repr__` → `"Book('Идиот', 'Достоевский', 1869)"`;
- `__eq__` — по `(title, author, year)`, для других типов `NotImplemented`;
- `__lt__` — по кортежу `(author, title)`, чтобы `sorted` расставлял книги по автору, а внутри автора — по названию.

**Что проверяется.** валидация в конструкторе; `__str__`/`__repr__`; сравнение кортежей в `__eq__` и `__lt__`.

**Пример.**

```python
b = Book("Идиот", "Достоевский", 1869)
print(b, [b], sep=" | ")   # Идиот — Достоевский (1869) | [Book('Идиот', 'Достоевский', 1869)]
print(b == Book("Идиот", "Достоевский", 1869), b.available)   # True True
```

In [ ]:
from tests_functions_oop import test_project_book

class Book:
    def __init__(self, title, author, year):
        ...

    def __str__(self):
        ...

    def __repr__(self):
        ...

    def __eq__(self, other):
        ...

    def __lt__(self, other):
        ...

test_project_book(Book)

### Компонент 3. Класс `Reader`

**Условие.** Напишите класс `Reader(name, max_books=3)`:

- проверки: `name` — непустая строка, `max_books` не меньше 1, иначе `ValueError`;
- атрибут `books` — список книг на руках (свой у каждого читателя);
- свойство `can_borrow` (через `@property`) — `True`, пока книг меньше `max_books`;
- `__len__` — число книг на руках;
- `__str__` → `"Читатель Аня: книг — 1"`.

**Что проверяется.** `@property`, вычисляемое из состояния; `__len__`; валидация.

**Пример.**

```python
r = Reader("Аня", max_books=1)
print(r.can_borrow, len(r))       # True 0
r.books.append(Book("Идиот", "Достоевский", 1869))
print(r.can_borrow, r)            # False Читатель Аня: книг — 1
```

In [ ]:
from tests_functions_oop import test_project_reader

class Reader:
    def __init__(self, name, max_books=3):
        ...

    @property
    def can_borrow(self):
        ...

    def __len__(self):
        ...

    def __str__(self):
        ...

test_project_reader(Reader, Book)

### Компонент 4. Класс `Library`

**Условие.** Напишите класс `Library(name)` — контейнер книг и читателей:

- атрибуты `books` (список) и `readers` (словарь `{имя: Reader}`);
- `add_books(self, *books)` — добавляет любое число книг;
- `register(self, reader)` — регистрирует читателя; если имя уже занято — `ValueError`;
- `find(self, predicate)` — список книг, для которых `predicate(book)` истинен;
- `find_by_title(self, title)` — книга с таким названием или `BookNotFoundError`;
- `lend(self, title, reader_name)` — выдать книгу: незарегистрированный читатель → `LibraryError`; книги нет → `BookNotFoundError`; книга занята → `BookNotAvailableError`; читатель достиг лимита (`not reader.can_borrow`) → `ReaderLimitError`. Если всё в порядке: `available = False`, книга добавляется в `reader.books`, метод возвращает книгу. Все проверки — **до** изменений;
- `return_book(self, title, reader_name)` — вернуть книгу: незарегистрированный читатель или книга, которой у него нет, → `LibraryError`; иначе книга убирается из `reader.books`, `available = True`, метод возвращает книгу;
- `available(self)` — **генератор** свободных книг в порядке добавления;
- `__len__` — число книг; `__contains__(self, title)` — есть ли книга с таким названием; `__iter__` — перебор всех книг.

**Что проверяется.** композиция двух видов объектов; `*args`; функция-предикат; генератор-метод; порядок проверок перед изменением состояния; своя иерархия исключений в деле.

**Ограничения.** `available` — генератор с `yield` (проверяется).

**Пример.**

```python
lib = Library("Городская")
lib.add_books(Book("Идиот", "Достоевский", 1869), Book("Бесы", "Достоевский", 1872))
lib.register(Reader("Аня"))
print(lib.lend("Идиот", "Аня"))            # Идиот — Достоевский (1869)
print([b.title for b in lib.available()])   # ['Бесы']
print(len(lib), "Идиот" in lib)              # 2 True
try:
    lib.lend("Идиот", "Аня")
except LibraryError as e:
    print("ошибка:", e)
```

In [ ]:
from tests_functions_oop import test_project_library

class Library:
    def __init__(self, name):
        ...

    def add_books(self, *books):
        ...

    def register(self, reader):
        ...

    def find(self, predicate):
        ...

    def find_by_title(self, title):
        ...

    def lend(self, title, reader_name):
        ...

    def return_book(self, title, reader_name):
        ...

    def available(self):
        ...

    def __len__(self):
        ...

    def __contains__(self, title):
        ...

    def __iter__(self):
        ...

test_project_library(Library, Book, Reader)

### Компонент 5. Отчёты и сценарий событий

**Условие.** Напишите три функции:

- `books_by_author(library)` — словарь `{автор: [названия]}` в порядке добавления книг (подсказка: `setdefault`);
- `top_readers(library, n=3)` — список пар `(имя, число книг на руках)` по убыванию числа книг, при равенстве — по алфавиту имён; читатели без книг тоже входят. `sorted` с `lambda`;
- `run_events(library, events)` — сценарий. `events` — список кортежей: `("add", {"title": ..., "author": ..., "year": ...})` — создать книгу из словаря (распаковка `Book(**data)`) и добавить; `("register", name)` — зарегистрировать `Reader(name)`; `("lend", title, name)`; `("return", title, name)`. Для каждого события в список результатов добавляется строка: `"добавлена: <title>"`, `"зарегистрирован: <name>"`, `"выдана: <title> → <name>"`, `"возвращена: <title>"`. Любая `LibraryError` или `ValueError` превращается в `"ошибка: <текст>"`, неизвестное событие — тоже ошибка. Обработка не прерывается. Используйте `try`/`except`/`else`.

**Что проверяется.** группировка через `setdefault`; `sorted` с составным ключом; `**kwargs`-распаковка словаря в конструктор; `try`/`except`/`else`; программа как последовательность событий.

**Ограничения.** в `top_readers` есть `lambda`, в `run_events` — `try`/`except` (проверяется).

**Пример.**

```python
lib = Library("Городская")
print(run_events(lib, [
    ("add", {"title": "Идиот", "author": "Достоевский", "year": 1869}),
    ("register", "Аня"),
    ("lend", "Идиот", "Аня"),
    ("lend", "Идиот", "Аня"),
]))
# ['добавлена: Идиот', 'зарегистрирован: Аня', 'выдана: Идиот → Аня', 'ошибка: книга «Идиот» уже выдана']
print(books_by_author(lib), top_readers(lib))   # {'Достоевский': ['Идиот']} [('Аня', 1)]
```

In [ ]:
from tests_functions_oop import test_project_reports

def books_by_author(library):
    ...


def top_readers(library, n=3):
    ...


def run_events(library, events):
    ...

test_project_reports(books_by_author, top_readers, run_events, Library, Book, Reader)

### Проект собран

Если все пять компонентов прошли тесты — у вас есть маленькая, но настоящая программа: модели
данных с проверками, контейнер с бизнес-логикой, собственные исключения, отчёты и сценарий.
Попробуйте расширить её самостоятельно, без тестов: добавьте `Book.__hash__`, чтобы книги можно
было класть в множество; срок возврата и генератор просроченных книг; сохранение библиотеки
в файл и загрузку из него.

# Разбор решений

Один из возможных вариантов решения каждой задачи — не единственно верный. Ячейки самодостаточны
и идут в том же порядке, что и задания; ячейки уровней зависят от предыдущих (Bank использует
BankAccount), поэтому выполняйте их сверху вниз. Классы уровней 1 и 2 приведены сразу в итоговом
виде — комментарии показывают, на каком шаге появилась каждая часть.

## Глава 4. Мини-задачи

**Мини-задача 1. rectangle_area.** Одна строка с `return`: функция отдаёт значение, а не печатает его.

In [ ]:
from tests_functions_oop import test_rectangle_area

def rectangle_area(width, height):
    return width * height

test_rectangle_area(rectangle_area)

**Мини-задача 2. make_counter.** `count` живёт во внешней функции; `nonlocal` разрешает вложенной функции менять её. Каждый вызов `make_counter()` создаёт новую `count`, поэтому счётчики независимы.

In [ ]:
from tests_functions_oop import test_make_counter

def make_counter():
    count = 0

    def counter():
        nonlocal count
        count += 1
        return count

    return counter

test_make_counter(make_counter)

**Мини-задача 3. add_bonus.** Списковое включение строит новый список; исходный не трогается.

In [ ]:
from tests_functions_oop import test_add_bonus

def add_bonus(scores, bonus):
    return [score + bonus for score in scores]

test_add_bonus(add_bonus)

**Мини-задача 4. join_words.** `*words` собирает слова в кортеж, `sep.join` склеивает; для пустого кортежа `join` возвращает пустую строку сам.

In [ ]:
from tests_functions_oop import test_join_words

def join_words(sep, *words):
    return sep.join(words)

test_join_words(join_words)

**Мини-задача 5. compose.** Внутренняя функция помнит `f` и `g` из замыкания и применяет их в нужном порядке.

In [ ]:
from tests_functions_oop import test_compose

def compose(f, g):
    def combined(x):
        return f(g(x))
    return combined

test_compose(compose)

**Мини-задача 6. sort_by_length.** Ключ-кортеж `(len(word), word)`: сначала сравнивается длина, при равенстве — само слово.

In [ ]:
from tests_functions_oop import test_sort_by_length

def sort_by_length(words):
    return sorted(words, key=lambda word: (len(word), word))

test_sort_by_length(sort_by_length)

**Мини-задача 7. sum_digits.** Базовый случай — однозначное число, рекурсивный шаг отделяет последнюю цифру.

In [ ]:
from tests_functions_oop import test_sum_digits

def sum_digits(n):
    if n < 10:                      # базовый случай: одна цифра
        return n
    return n % 10 + sum_digits(n // 10)   # последняя цифра + сумма остальных

test_sum_digits(sum_digits)

**Мини-задача 8. double_result.** Обёртка принимает `*args, **kwargs`, вызывает `func` и удваивает результат; `@wraps(func)` сохраняет имя.

In [ ]:
from tests_functions_oop import test_double_result

from functools import wraps

def double_result(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

test_double_result(double_result)

**Мини-задача 9. evens_up_to.** `range(0, n + 1, 2)` даёт чётные числа; для `n < 0` диапазон пуст. `yield` делает функцию генератором.

In [ ]:
from tests_functions_oop import test_evens_up_to

def evens_up_to(n):
    for number in range(0, n + 1, 2):
        yield number

test_evens_up_to(evens_up_to)

## Глава 4. Десять заданий

**Задача 1. clamp.** `min(value, high)` срезает сверху, `max(low, ...)` — снизу.

In [ ]:
from tests_functions_oop import test_clamp

def clamp(value, low, high):
    return max(low, min(value, high))

test_clamp(clamp)

**Задача 2. snake_to_camel.** Приводим к нижнему регистру, режем по `_`, первое слово оставляем, остальные через `capitalize`.

In [ ]:
from tests_functions_oop import test_snake_to_camel

def snake_to_camel(text):
    words = text.lower().split("_")
    return words[0] + "".join(word.capitalize() for word in words[1:])

test_snake_to_camel(snake_to_camel)

**Задача 3. chunk.** `range(0, len(items), size)` даёт начала кусков, срез `items[start:start + size]` — сам кусок (последний короче автоматически).

In [ ]:
from tests_functions_oop import test_chunk

def chunk(items, size):
    result = []
    for start in range(0, len(items), size):
        result.append(items[start:start + size])
    return result

test_chunk(chunk)

**Задача 4. make_email.** Вся работа — в сигнатуре с умолчаниями и одной f-строке.

In [ ]:
from tests_functions_oop import test_make_email

def make_email(first, last, domain="example.com", separator="."):
    return f"{first.lower()}{separator}{last.lower()}@{domain}"

test_make_email(make_email)

**Задача 5. average.** `nums` — кортеж; пустой кортеж — ложь, поэтому `if not nums`.

In [ ]:
from tests_functions_oop import test_average

def average(*nums):
    if not nums:
        return 0.0
    return sum(nums) / len(nums)

test_average(average)

**Задача 6. make_tag.** `attrs.items()` сохраняет порядок передачи; каждая пара превращается в `key="value"`.

In [ ]:
from tests_functions_oop import test_make_tag

def make_tag(name, **attrs):
    parts = [name] + [f'{key}="{value}"' for key, value in attrs.items()]
    return "<" + " ".join(parts) + ">"

test_make_tag(make_tag)

**Задача 7. apply_pipeline.** Цикл по функциям, результат каждой становится входом следующей.

In [ ]:
from tests_functions_oop import test_apply_pipeline

def apply_pipeline(value, *funcs):
    for func in funcs:
        value = func(value)
    return value

test_apply_pipeline(apply_pipeline)

**Задача 8. even_squares_desc.** Конвейер `filter` → `map` → `sorted`; `sorted` принимает итератор и возвращает список.

In [ ]:
from tests_functions_oop import test_even_squares_desc

def even_squares_desc(nums):
    evens = filter(lambda x: x % 2 == 0, nums)
    squares = map(lambda x: x * x, evens)
    return sorted(squares, reverse=True)

test_even_squares_desc(even_squares_desc)

**Задача 9. flatten.** Для вложенного списка — рекурсивный вызов и `extend`, для обычного элемента — `append`.

In [ ]:
from tests_functions_oop import test_flatten

def flatten(nested):
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))    # рекурсивный шаг: вложенный список
        else:
            result.append(item)             # базовый случай: обычный элемент
    return result

test_flatten(flatten)

**Задача 10. memoize + fib.** Кэш живёт в замыкании обёртки; ключ — кортеж `args`. `fib` при первом вызове заполняет кэш снизу вверх, и каждое значение считается один раз.

In [ ]:
from tests_functions_oop import test_memoize_fib

from functools import wraps

def memoize(func):
    cache = {}                      # замыкание: у каждой обёрнутой функции свой кэш

    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    return wrapper


@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

test_memoize_fib(memoize, fib)

## Глава 5. Мини-задачи

**Мини-задача 10. Point.** `distance_to` принимает другой объект и читает его поля; `move` меняет `self`.

In [ ]:
from tests_functions_oop import test_point

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance_to(self, other):
        return ((self.x - other.x) ** 2 + (self.y - other.y) ** 2) ** 0.5

    def move(self, dx, dy):
        self.x += dx
        self.y += dy

test_point(Point)

**Мини-задача 11. Song и Playlist.** Плейлист хранит объекты `Song`; `longest` — `max` с ключом по длительности и отдельная ветка для пустого списка.

In [ ]:
from tests_functions_oop import test_playlist

class Song:
    def __init__(self, title, duration):
        self.title = title
        self.duration = duration


class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []

    def add(self, song):
        self.songs.append(song)

    def total_duration(self):
        return sum(song.duration for song in self.songs)

    def longest(self):
        if not self.songs:
            return None
        return max(self.songs, key=lambda song: song.duration)

test_playlist(Playlist, Song)

**Мини-задача 12. Vector.** `__add__` создаёт новый вектор, `__eq__` возвращает `NotImplemented` для чужих типов.

In [ ]:
from tests_functions_oop import test_vector

class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

test_vector(Vector)

**Мини-задача 13. Shape, Rectangle, Circle.** `describe` в базовом классе вызывает `self.area()` — метод того класса, каким создан объект; подклассы переопределяют только `area`.

In [ ]:
from tests_functions_oop import test_shapes

import math


class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        raise NotImplementedError("area() должен быть переопределён в подклассе")

    def describe(self):
        return f"{self.name}: площадь {self.area():.2f}"


class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__("прямоугольник")
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        super().__init__("круг")
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

test_shapes(Shape, Rectangle, Circle)

**Мини-задача 14. safe_sqrt.** Своё исключение — пустой класс от `ValueError`; сообщение формируется в `raise`.

In [ ]:
from tests_functions_oop import test_safe_sqrt

class NegativeNumberError(ValueError):
    """Попытка извлечь корень из отрицательного числа."""


def safe_sqrt(x):
    if x < 0:
        raise NegativeNumberError(f"нельзя извлечь корень из отрицательного числа: {x}")
    return x ** 0.5

test_safe_sqrt(safe_sqrt, NegativeNumberError)

**Мини-задача 15. read_numbers.** `int(text)` внутри `try`; `except ValueError: pass` пропускает мусор, не скрывая другие ошибки.

In [ ]:
from tests_functions_oop import test_read_numbers

def read_numbers(strings):
    numbers = []
    for text in strings:
        try:
            numbers.append(int(text))
        except ValueError:
            pass                      # не число — просто пропускаем
    return numbers

test_read_numbers(read_numbers)

**Мини-задача 16. модуль shapes.py.** Первая ячейка записывает файл, вторая проверяет: импорт молчит благодаря `if __name__ == "__main__"`.

In [ ]:
%%writefile shapes.py
"""Площади простых фигур."""
import math


def rectangle_area(width, height):
    return width * height


def circle_area(radius):
    return math.pi * radius ** 2


if __name__ == "__main__":
    # выполняется только при запуске файла напрямую, но не при импорте
    print("прямоугольник 3x4:", rectangle_area(3, 4))
    print("круг r=1:", circle_area(1))

In [ ]:
from tests_functions_oop import test_shapes_module

test_shapes_module()

## Уровень 1. BankAccount

**Итоговый класс BankAccount (шаги 1–7).** Проверки собраны в `_check_amount`; `withdraw` сравнивает сумму с `available()`, а не с `balance` — это даст кредитному счёту уйти в минус без изменения `withdraw`. `transfer` не трогает `balance` напрямую, поэтому проверки и история работают сами. Метод `month_end` добавлен для уровня 3.

In [ ]:
from tests_functions_oop import test_account_step1, test_account_step2, test_account_step3, test_account_step4, test_account_step5, test_account_step6, test_account_step7

class InsufficientFundsError(Exception):            # шаг 5
    """Недостаточно средств на счёте."""


class BankAccount:
    currency = "RUB"                                   # шаг 2: атрибут класса, общий для всех счетов

    def __init__(self, owner, balance=0):              # шаги 1–2
        if not isinstance(owner, str) or not owner.strip():        # шаг 5
            raise ValueError("имя владельца не может быть пустым")
        if balance < 0:
            raise ValueError("начальный баланс не может быть отрицательным")
        self.owner = owner
        self.balance = balance
        self.history = []                              # шаг 2: у каждого счёта свой список

    # ---------- шаг 5: проверки ----------
    def _check_amount(self, amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("сумма должна быть числом")
        if amount <= 0:
            raise ValueError("сумма должна быть положительной")

    def available(self):
        return self.balance                            # сколько можно снять (пригодится на уровне 3)

    # ---------- шаги 3–4: операции и история ----------
    def deposit(self, amount):
        self._check_amount(amount)
        self.balance += amount
        self.history.append(("deposit", amount))
        return self.balance

    def withdraw(self, amount):
        self._check_amount(amount)
        if amount > self.available():
            raise InsufficientFundsError(
                f"недостаточно средств: доступно {self.available()}, запрошено {amount}")
        self.balance -= amount
        self.history.append(("withdraw", amount))
        return self.balance

    def total_deposited(self):
        return sum(amount for operation, amount in self.history if operation == "deposit")

    def total_withdrawn(self):
        return sum(amount for operation, amount in self.history if operation == "withdraw")

    # ---------- шаг 6: магические методы ----------
    def __str__(self):
        return f"{self.owner}: {self.balance} {self.currency}"

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, {self.balance})"

    def __eq__(self, other):
        if not isinstance(other, BankAccount):
            return NotImplemented
        return self.owner == other.owner and self.balance == other.balance

    def __lt__(self, other):
        return self.balance < other.balance

    # ---------- шаг 7: переводы и __len__ ----------
    def transfer(self, other, amount):
        self.withdraw(amount)                          # если денег нет — исключение, ничего не меняется
        other.deposit(amount)

    def __len__(self):
        return len(self.history)

    # ---------- уровень 3, шаг 1 ----------
    def month_end(self):
        return 0                                       # у обычного счёта в конце месяца ничего не происходит

test_account_step1(BankAccount)
test_account_step2(BankAccount)
test_account_step3(BankAccount)
test_account_step4(BankAccount)
test_account_step5(BankAccount, InsufficientFundsError)
test_account_step6(BankAccount)
test_account_step7(BankAccount)

## Уровень 2. Bank

**Итоговый класс Bank (шаги 1–5).** Счета лежат в словаре по номеру, `_next_number` никогда не уменьшается. `__iter__` — генератор по отсортированным номерам, и все остальные методы (`total_balance`, `richest`, `find`, `sorted_by_balance`) перебирают `self`, а не словарь. `add_account` и `month_end` добавлены для уровня 3; `open_account` выражен через `add_account`.

In [ ]:
from tests_functions_oop import test_bank_step1, test_bank_step2, test_bank_step3, test_bank_step4, test_bank_step5

class AccountNotFoundError(Exception):               # шаг 2
    """Счёт с таким номером не найден."""


class Bank:
    def __init__(self, name):                          # шаг 1
        self.name = name
        self.accounts = {}                             # номер → BankAccount
        self._next_number = 1

    def open_account(self, owner, balance=0):          # шаг 1
        return self.add_account(BankAccount(owner, balance))

    # ---------- шаг 2: поиск и закрытие ----------
    def get_account(self, number):
        if number not in self.accounts:
            raise AccountNotFoundError(f"счёт №{number} не найден")
        return self.accounts[number]

    def close_account(self, number):
        account = self.get_account(number)
        if account.balance != 0:
            raise ValueError("нельзя закрыть счёт с ненулевым балансом")
        del self.accounts[number]

    # ---------- шаг 3: магические методы контейнера ----------
    def __len__(self):
        return len(self.accounts)

    def __contains__(self, number):
        return number in self.accounts

    def __getitem__(self, number):
        return self.get_account(number)

    def __iter__(self):
        for number in sorted(self.accounts):
            yield self.accounts[number]

    def __str__(self):
        return f"Банк {self.name}: счетов — {len(self)}"

    # ---------- шаг 4: свойство, поиск, сортировка ----------
    @property
    def total_balance(self):
        return sum(account.balance for account in self)

    def richest(self):
        if not self.accounts:
            return None
        return max(self, key=lambda account: account.balance)

    def find(self, predicate):
        return [account for account in self if predicate(account)]

    def sorted_by_balance(self, descending=False):
        return sorted(self, key=lambda account: account.balance, reverse=descending)

    # ---------- шаг 5: операции по номеру ----------
    def deposit(self, number, amount):
        return self.get_account(number).deposit(amount)

    def withdraw(self, number, amount):
        return self.get_account(number).withdraw(amount)

    def transfer(self, from_number, to_number, amount):
        source = self.get_account(from_number)
        target = self.get_account(to_number)
        source.transfer(target, amount)

    # ---------- уровень 3, шаг 2 ----------
    def add_account(self, account):
        if not isinstance(account, BankAccount):
            raise TypeError("можно добавить только объект BankAccount или его подкласса")
        number = self._next_number
        self.accounts[number] = account
        self._next_number += 1
        return number

    def month_end(self):
        return [account.month_end() for account in self]

test_bank_step1(Bank)
test_bank_step2(Bank, AccountNotFoundError)
test_bank_step3(Bank)
test_bank_step4(Bank)
test_bank_step5(Bank)

## Уровень 3

**Шаг 1. SavingsAccount и CreditAccount.** Оба подкласса вызывают `super().__init__` и добавляют своё поле. `CreditAccount` переопределяет только `available` — `withdraw` родителя сам разрешает минус до лимита.

In [ ]:
from tests_functions_oop import test_level3_step1

class SavingsAccount(BankAccount):
    def __init__(self, owner, balance=0, rate=0.05):
        super().__init__(owner, balance)               # всё, что умеет BankAccount, остаётся
        self.rate = rate

    def add_interest(self):
        interest = round(self.balance * self.rate, 2)
        if interest > 0:
            self.deposit(interest)                     # проценты — это обычное пополнение
        return interest

    def month_end(self):                               # переопределяем поведение родителя
        return self.add_interest()


class CreditAccount(BankAccount):
    fee_rate = 0.02

    def __init__(self, owner, balance=0, credit_limit=1000):
        super().__init__(owner, balance)
        self.credit_limit = credit_limit

    def available(self):                               # withdraw родителя сам вызовет этот метод
        return self.balance + self.credit_limit

    def month_end(self):
        if self.balance >= 0:
            return 0
        fee = round(-self.balance * self.fee_rate, 2)
        self.balance -= fee
        self.history.append(("fee", fee))
        return fee

test_level3_step1(BankAccount, SavingsAccount, CreditAccount)

**Шаг 2. Bank принимает любые счета.** `add_account` и полиморфный `month_end` уже включены в итоговый `Bank` выше — здесь только проверка.

In [ ]:
from tests_functions_oop import test_level3_step2

test_level3_step2(Bank, SavingsAccount, CreditAccount)

**Шаг 3. Генератор операций и отчёты.** `iter_operations` — два вложенных цикла с `yield`; `top_accounts` сортирует пары `(номер, счёт)` по ключу `(-balance, номер)`; `operations_summary` потребляет генератор.

In [ ]:
from tests_functions_oop import test_level3_step3

def iter_operations(bank):
    for number in sorted(bank.accounts):
        for operation, amount in bank.accounts[number].history:
            yield (number, operation, amount)


def top_accounts(bank, n=3):
    ranked = sorted(bank.accounts.items(), key=lambda pair: (-pair[1].balance, pair[0]))
    return [account.owner for number, account in ranked[:n]]


def operations_summary(bank):
    summary = {}
    for number, operation, amount in iter_operations(bank):
        summary[operation] = summary.get(operation, 0) + amount
    return summary

test_level3_step3(iter_operations, top_accounts, operations_summary, Bank)

**Шаг 4. catch_errors и run_commands.** Декоратор превращает исключение в строку; словарь методов и распаковка `*args` избавляют от цепочки `if`/`elif`.

In [ ]:
from tests_functions_oop import test_level3_step4

from functools import wraps


def catch_errors(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as error:
            return f"ошибка: {error}"
    return wrapper


def run_commands(bank, commands):
    handlers = {                                       # функции — обычные значения словаря
        "open": bank.open_account,
        "deposit": bank.deposit,
        "withdraw": bank.withdraw,
        "transfer": bank.transfer,
        "balance": lambda number: bank[number].balance,
    }
    results = []
    for name, *args in commands:
        handler = handlers.get(name)
        if handler is None:
            results.append(f"ошибка: неизвестная команда {name}")
            continue
        results.append(str(catch_errors(handler)(*args)))
    return results

test_level3_step4(catch_errors, run_commands, Bank)

## Итоговый проект

**Компонент 1. Исключения.** Пустые классы с docstring: вся ценность — в иерархии.

In [ ]:
from tests_functions_oop import test_project_exceptions

class LibraryError(Exception):
    """Базовая ошибка библиотеки: ловим её, когда неважно, что именно случилось."""


class BookNotFoundError(LibraryError):
    """Книги с таким названием нет."""


class BookNotAvailableError(LibraryError):
    """Книга уже выдана другому читателю."""


class ReaderLimitError(LibraryError):
    """Читатель уже взял максимум книг."""

test_project_exceptions(LibraryError, BookNotFoundError, BookNotAvailableError, ReaderLimitError)

**Компонент 2. Book.** Валидация в конструкторе; `__eq__` и `__lt__` сравнивают кортежи полей.

In [ ]:
from tests_functions_oop import test_project_book

class Book:
    def __init__(self, title, author, year):
        if not isinstance(title, str) or not title.strip():
            raise ValueError("название книги не может быть пустым")
        if not isinstance(author, str) or not author.strip():
            raise ValueError("имя автора не может быть пустым")
        if not isinstance(year, int) or not 1450 <= year <= 2100:
            raise ValueError("год должен быть целым числом от 1450 до 2100")
        self.title = title
        self.author = author
        self.year = year
        self.available = True

    def __str__(self):
        return f"{self.title} — {self.author} ({self.year})"

    def __repr__(self):
        return f"Book({self.title!r}, {self.author!r}, {self.year})"

    def __eq__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return (self.title, self.author, self.year) == (other.title, other.author, other.year)

    def __lt__(self, other):
        return (self.author, self.title) < (other.author, other.title)

test_project_book(Book)

**Компонент 3. Reader.** `can_borrow` — свойство, вычисляемое из длины списка; состояния-дубликата нет.

In [ ]:
from tests_functions_oop import test_project_reader

class Reader:
    def __init__(self, name, max_books=3):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("имя читателя не может быть пустым")
        if max_books < 1:
            raise ValueError("лимит книг должен быть не меньше 1")
        self.name = name
        self.max_books = max_books
        self.books = []

    @property
    def can_borrow(self):
        return len(self.books) < self.max_books

    def __len__(self):
        return len(self.books)

    def __str__(self):
        return f"Читатель {self.name}: книг — {len(self)}"

test_project_reader(Reader, Book)

**Компонент 4. Library.** В `lend` все проверки стоят до изменения состояния; `available` — генератор; `__iter__` отдаёт `iter(self.books)`.

In [ ]:
from tests_functions_oop import test_project_library

class Library:
    def __init__(self, name):
        self.name = name
        self.books = []                                # композиция: библиотека владеет книгами
        self.readers = {}                              # имя → Reader

    def add_books(self, *books):
        self.books.extend(books)

    def register(self, reader):
        if reader.name in self.readers:
            raise ValueError(f"читатель {reader.name} уже зарегистрирован")
        self.readers[reader.name] = reader

    def find(self, predicate):
        return [book for book in self.books if predicate(book)]

    def find_by_title(self, title):
        for book in self.books:
            if book.title == title:
                return book
        raise BookNotFoundError(f"книга «{title}» не найдена")

    def lend(self, title, reader_name):
        if reader_name not in self.readers:
            raise LibraryError(f"читатель {reader_name} не зарегистрирован")
        reader = self.readers[reader_name]
        book = self.find_by_title(title)
        if not book.available:
            raise BookNotAvailableError(f"книга «{title}» уже выдана")
        if not reader.can_borrow:
            raise ReaderLimitError(f"{reader_name} уже взял максимум книг ({reader.max_books})")
        book.available = False                         # все проверки пройдены — меняем состояние
        reader.books.append(book)
        return book

    def return_book(self, title, reader_name):
        if reader_name not in self.readers:
            raise LibraryError(f"читатель {reader_name} не зарегистрирован")
        reader = self.readers[reader_name]
        for book in reader.books:
            if book.title == title:
                reader.books.remove(book)
                book.available = True
                return book
        raise LibraryError(f"у читателя {reader_name} нет книги «{title}»")

    def available(self):
        for book in self.books:
            if book.available:
                yield book

    def __len__(self):
        return len(self.books)

    def __contains__(self, title):
        return any(book.title == title for book in self.books)

    def __iter__(self):
        return iter(self.books)

test_project_library(Library, Book, Reader)

**Компонент 5. Отчёты и сценарий.** `Book(**data)` распаковывает словарь события; `try`/`except`/`else` отделяет успешный путь от ошибочного, и обработка событий не прерывается.

In [ ]:
from tests_functions_oop import test_project_reports

def books_by_author(library):
    by_author = {}
    for book in library:
        by_author.setdefault(book.author, []).append(book.title)
    return by_author


def top_readers(library, n=3):
    ranked = sorted(library.readers.values(), key=lambda reader: (-len(reader), reader.name))
    return [(reader.name, len(reader)) for reader in ranked[:n]]


def run_events(library, events):
    results = []
    for kind, *args in events:
        try:
            if kind == "add":
                book = Book(**args[0])                 # словарь события распаковывается в параметры Book
                library.add_books(book)
                message = f"добавлена: {book.title}"
            elif kind == "register":
                library.register(Reader(*args))
                message = f"зарегистрирован: {args[0]}"
            elif kind == "lend":
                book = library.lend(*args)
                message = f"выдана: {book.title} → {args[1]}"
            elif kind == "return":
                book = library.return_book(*args)
                message = f"возвращена: {book.title}"
            else:
                raise LibraryError(f"неизвестное событие {kind}")
        except (LibraryError, ValueError, TypeError) as error:
            results.append(f"ошибка: {error}")
        else:
            results.append(message)
    return results

test_project_reports(books_by_author, top_readers, run_events, Library, Book, Reader)

# Что я теперь умею

Пройдите по списку и отметьте, что действительно получается сделать без подсказок.
Рядом — где это отрабатывалось.

- [ ] **умею писать функции** с параметрами и `return` — мини-задача 1, задачи 1–3;
- [ ] **умею работать с `*args` и `**kwargs`** и распаковкой при вызове — мини-задача 4, задачи 5–6, `run_commands`, `run_events`;
- [ ] **понимаю функции высших порядков**: передаю и возвращаю функции — мини-задача 5, задача 7, `Bank.find`;
- [ ] **понимаю `lambda`** и использую её с `sorted`, `map`, `filter` — мини-задача 6, задача 8, `top_accounts`, `top_readers`;
- [ ] **умею писать рекурсивные функции** с базовым случаем — мини-задача 7, задача 9, `fib`;
- [ ] **понимаю декораторы**: обёртка, `@`, `functools.wraps` — мини-задача 8, задача 10, `catch_errors`;
- [ ] **умею писать генераторы** с `yield` — мини-задача 9, `Bank.__iter__`, `iter_operations`, `Library.available`;
- [ ] **умею создавать классы** с полями и методами — мини-задачи 10–11, `BankAccount`;
- [ ] **понимаю `self` и `__init__`** и разницу между атрибутами класса и экземпляра — `BankAccount`, шаги 1–2;
- [ ] **умею использовать магические методы**: `__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__getitem__`, `__iter__` — мини-задача 12, шаг 6, `Bank`, `Book`;
- [ ] **понимаю наследование и полиморфизм**, `super()` и переопределение — мини-задача 13, `SavingsAccount`, `CreditAccount`, `Bank.month_end`;
- [ ] **умею работать с исключениями**: `try`/`except`/`else`/`finally`, `raise`, свои классы — мини-задачи 14–15, шаг 5, исключения проекта;
- [ ] **умею разбивать код на модули** и понимаю `if __name__ == "__main__"` — мини-задача 16, структура компонентов проекта;
- [ ] **умею собирать несколько классов в одну программу** — уровень 3, итоговый проект.

Если какой-то пункт не отмечен — вернитесь к указанной задаче, удалите своё решение и напишите
его заново, не подглядывая в разбор. Умение появляется на втором-третьем самостоятельном повторе.
